# LangChain 1.0 Agent 实践

## 模型 · 工具 · 记忆 · 中间件 · 综合实战

&emsp;&emsp;本文档面向 LangChain 1.0 Agent 入门教学，按真实开发路径组织内容：先掌握模型调用，再学习 Agent 框架、工具接入、记忆管理和中间件，最后完成一个可以查新闻、查天气、操作文件的综合 Agent。


## 项目介绍

&emsp;&emsp;本项目是一个面向教学的 LangChain Agent 实践案例，目标不是只展示零散 API，而是帮助学习者理解一个 Agent 项目从模型、工具到运行时能力的完整组成。

本 notebook 会围绕以下问题展开：

1. 如何用 LangChain 统一接入大模型，并完成基础调用、流式输出、批处理和结构化输出。
2. 如何使用 `create_agent` 创建 LangChain 1.0 Agent，并理解 Agent Runtime 的运行方式。
3. 如何接入不同来源的工具，包括 `TavilySearch`、自定义 Python 工具和 MCP 工具服务。
4. 如何用 ReAct 思路理解 Agent 的“思考 - 行动 - 观察 - 回答”循环。
5. 如何用 `checkpointer` 管理多轮对话记忆。
6. 如何用中间件处理模型动态选择、消息压缩和人在闭环审批。
7. 如何把新闻搜索、天气查询和文件操作组合成一个综合 Agent。

&emsp;&emsp;课程中使用的两个 MCP 服务已经放在当前目录：

| 服务 | 目录 | 作用 |
| --- | --- | --- |
| `OpenWeatherMCPServer` | `mcp/01_weather-mcp-server/` | 通过 OpenWeather API 查询实时天气 |
| `SecureFileMCPServer` | `mcp/02_file-mcp-server/` | 在受限目录内列文件、读文件、查看文件信息和执行受控重命名 |


## 目录与学习路径

1. **模型基础与模型调用**：从模型接入、消息格式、流式响应、批处理、结构化输出到流式聊天机器人。

2. **全新 Agent 框架介绍**：理解 LangChain 1.0 的 `create_agent`、Agent Runtime、模型接入方式和运行结构。

3. **工具：内置工具、自定义工具与 MCP**：接入 `TavilySearch`、自定义天气工具和 MCP 工具，并用 ReAct 理解 Agent 的思考-行动循环。

4. **记忆管理**：学习如何让 Agent 在多轮对话中保留、恢复和管理上下文。

5. **中间件机制**：学习模型动态选择、消息压缩以及 Human-in-the-loop 人在闭环。

6. **综合实战：新闻 + 天气 + 文件 Agent**：把模型、工具、MCP、ReAct、记忆和安全约束组合成一个可运行的综合 Agent。

> 说明：原 Part 1、Part 2、Part 3 文件仍保留不变；本 notebook 是重新编排后的整合版。

> 说明：本 notebook 只在关键执行节点打印 `【要点】` 日志，用于提示当前操作的教学重点；普通查看变量、拆分展示和补充说明类 cell 不额外打印，避免输出区过于嘈杂。


# 第一章 模型基础与模型调用

&emsp;&emsp;本章整合原 `Part 1.LangChain Agent 模型篇.ipynb`，保留模型接入、统一消息格式、流式响应、批处理、结构化输出和流式聊天机器人实践等内容。


> 课程环境说明：本 notebook 已按 `langchain==1.3.9` 调试通过，建议使用前文创建的 `env_agent` Conda 环境运行。


<div align=center><img src="https://muyu001.oss-cn-beijing.aliyuncs.com/img/langchain_arch.png" width=60%></div>

## 模型接入

### 1. DeepSeek 接入 LangChain 流程

#### 1.1 环境检查

&emsp;&emsp;不同依赖版本的 API 行为可能存在差异。为了保证课堂代码可以复现，建议使用 `requirements.txt` 中固定的版本。本课程使用 `langchain==1.3.9`，可以通过下面的代码检查当前环境中的 LangChain 版本：


In [ ]:
import langchain
langchain.__version__

#### 1.2 DeepSeek API 连通性测试

&emsp;&emsp;在进行`LangChain`开发之前，首先需要准备一个可以调用的大模型。这里选择使用`DeepSeek`官方模型，并通过`DeepSeek API_KEY`进行调用。如果初次使用，需要先在`DeepSeek`官网注册并创建新的`API_KEY`，官方地址为：https://platform.deepseek.com/usage

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202506091257041.png" width=60%></div>

&emsp;&emsp;注册好 `DeepSeek` 的 `API_KEY` 后，在当前 notebook 所在目录下创建一个 `.env` 文件，用于保存密钥。不要把真实 API Key 直接写进 notebook 或提交到 Git 仓库。示例格式如下：

```bash
DEEPSEEK_API_KEY=你的 DeepSeek Key
```


<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/202506101758896.png" alt="image-20250610175750011" style="zoom:50%;" />

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/202506101759417.png" alt="image-20250610175921388" style="zoom:50%;" />

&emsp;&emsp;我们在当前的运行环境下不使用`LangChain`，直接使用`DeepSeek`的`API`进行网络连通性测试，测试代码如下：

In [ ]:
print("【要点】先绕开 LangChain，直接调用 DeepSeek 原生 API，确认 API Key 和网络连通性。")

import os
import httpx
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from openai import OpenAI

# 加载环境变量
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

DeepSeek_API_KEY = os.getenv("DEEPSEEK_API_KEY")
# print(DeepSeek_API_KEY)  # 可以通过打印查看

# 清理系统代理环境变量，避免本地代理配置影响 API 请求
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy",
]:
    os.environ.pop(k, None)

# 初始化 DeepSeek 的 API 客户端
client = OpenAI(api_key=DeepSeek_API_KEY, base_url="https://api.deepseek.com")

# 调用 DeepSeek 的 API，生成回答
response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role": "system", "content": "你是乐于助人的助手，请根据用户的问题给出回答"},
        {"role": "user", "content": "你好，请你介绍一下你自己。"},
    ],
)

# 打印模型最终的响应结果
print(response.choices[0].message.content)

&emsp;&emsp;如果可以正常收到`DeepSeek`模型的响应，则说明`DeepSeek`的`API`已经可以正常使用且网络连通性正常。

#### 1.3 DeepSeek 接入 LangChain

&emsp;&emsp;完成`DeepSeek API`连通性测试后，接下来将`DeepSeek`接入`LangChain`。这里使用`LangChain`官方生态中的`langchain-deepseek`集成包，它可以直接复用相同的`DeepSeek API_KEY`与模型进行交互。首先安装依赖包：

In [ ]:
# ! pip install langchain-deepseek

&emsp;&emsp;安装好`langchain-deepseek`后，可以通过`ChatDeepSeek`初始化大模型：

In [ ]:
print("【要点】使用 langchain-deepseek 的 ChatDeepSeek，把厂商模型接入 LangChain。")
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-v4-flash")

&emsp;&emsp;其中`model`用来指定要使用的模型名称。`deepseek-chat`适合通用对话任务，`deepseek-reasoner`适合需要推理过程的任务。

In [ ]:
question = "你好，请你介绍一下你自己。"

result = model.invoke(question)
print(result.content)

In [ ]:
result

In [ ]:
model = ChatDeepSeek(model="deepseek-reasoner")

result = model.invoke(question)
print(result.content)

In [ ]:
result

In [ ]:
result.additional_kwargs

In [ ]:
result.additional_kwargs['reasoning_content']

&emsp;&emsp;至此，我们已经完成了`DeepSeek`从原生`API`调用到`LangChain`接入的完整流程。使用专用集成包的优点是代码简洁、语义清晰，适合课程中演示某个具体模型厂商的标准接入方式。

### 2. 兼容 OpenAI API 模型服务的最通用方法

&emsp;&emsp;除了安装各模型厂商对应的专用集成包外，更通用的方法是使用 `init_chat_model` 接入兼容 OpenAI API 格式的模型服务。很多模型平台和企业内部网关都提供 `base_url + api_key + model` 这种调用方式，例如 DeepSeek、SiliconFlow、私有化模型服务、统一模型网关或反向代理服务等。

&emsp;&emsp;只要接口兼容 OpenAI Chat Completions 协议，就可以用同一套 LangChain 代码接入。后续切换模型时，通常只需要替换 `base_url`、`api_key` 和 `model` 名称。因此，这一节重点掌握“统一模型入口”的思路，而不是记住某一家厂商的固定写法。


下面以 DeepSeek 的 `deepseek-v4-flash` 为例进行演示：


In [ ]:
print("【要点】用 init_chat_model 接入 DeepSeek 这类 OpenAI 兼容服务：重点看 base_url、api_key、model 三个参数。")
import os
import httpx
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model


# 清理系统代理环境变量，避免本地代理配置影响 API 请求
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy",
]:
    os.environ.pop(k, None)

# 1. 加载环境变量
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)
api_key = os.getenv("DEEPSEEK_API_KEY")

# 2. 初始化大语言模型
my_llm = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url="https://api.deepseek.com",
    api_key=api_key,
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

下面以 SiliconFlow 上的 `Qwen/Qwen3-8B` 为例进行演示。可以看到，代码结构基本不变，只替换模型名、服务地址和 API Key 环境变量：


In [ ]:
print("【要点】同一套 init_chat_model 代码切换到 SiliconFlow，只需要替换模型名、服务地址和 Key。")
import os
import httpx
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model


# 清理系统代理环境变量，避免本地代理配置影响 API 请求
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy",
]:
    os.environ.pop(k, None)

# 1. 加载环境变量
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)
api_key = os.getenv("SILICONFLOW_API_KEY")

# 2. 初始化大语言模型
my_llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key,
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

## 模型调用

&emsp;&emsp;在 LangChain 1.0 中，Agent 教学的主线从“手动拼接链式流程”转向“模型、工具、状态和运行时的统一管理”。LCEL 仍然可以用于链式组合，但本课程会优先使用模型标准调用、`create_agent`、工具调用、记忆管理和中间件来讲解 Agent 开发。

&emsp;&emsp;本章后续示例统一通过 `create_model()` 创建模型。该函数封装了四种 OpenAI 兼容模型选择：DeepSeek 的 `deepseek-v4-flash`、DeepSeek 的 `deepseek-v4-pro`、SiliconFlow 上的 `Qwen/Qwen3-8B`，以及 OpenRouter 上的 `qwen/qwen3-coder:free`。默认使用 `deepseek-v4-flash`，也可以在调用时手动传入模型名。


In [ ]:
print("【要点】统一封装 create_model()，后续所有模型调用和 Agent 都复用这一入口。")
import os
import httpx
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

# 清理系统代理环境变量，避免本地代理配置影响 API 请求
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy",
]:
    os.environ.pop(k, None)

# 如果你的 .env 就在当前目录，也可以直接使用 load_dotenv(override=True)
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

MODEL_CONFIGS = {
    "deepseek-v4-flash": {
        "model": "deepseek-v4-flash",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的轻量快速模型",
    },
    "deepseek-v4-pro": {
        "model": "deepseek-v4-pro",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的更强能力模型",
    },
    "vllm-qwen122b": {
        "model": os.getenv("vLLM_QWEN122B_MODEL_ID"),
        "base_url": os.getenv("vLLM_QWEN122B_BASE_URL"),
        "api_key_env": "vLLM_QWEN122B_API_KEY",
        "description": "vLLM 部署的 Qwen 122B 模型（默认）",
    },
    "Qwen/Qwen3-8B": {
        "model": "Qwen/Qwen3-8B",
        "base_url": "https://api.siliconflow.cn/v1",
        "api_key_env": "SILICONFLOW_API_KEY",
        "description": "SiliconFlow 上的 Qwen/Qwen3-8B 模型",
    },
    "qwen/qwen3-coder:free": {
        "model": "qwen/qwen3-coder:free",
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
        "description": "OpenRouter 上的 Qwen3 Coder 免费模型",
    },
}

DEFAULT_MODEL_NAME = "vllm-qwen122b"
# 切换到 DeepSeek：create_model("deepseek-v4-flash") 或 create_model("deepseek-v4-pro")


def create_model(
    model_name: str = DEFAULT_MODEL_NAME,
    temperature: float = 0.3,
    max_tokens: int = 10000,
    verbose: bool = True,
):
    """创建课程统一使用的 OpenAI 兼容聊天模型。

    支持五种选择：
    - vllm-qwen122b：从 .env 读取 vLLM Qwen 122B 配置（默认）
    - deepseek-v4-flash：DeepSeek 兼容 OpenAI API 模型服务
    - deepseek-v4-pro：DeepSeek 兼容 OpenAI API 模型服务
    - Qwen/Qwen3-8B：SiliconFlow 上的 Qwen 模型
    - qwen/qwen3-coder:free：OpenRouter 上的 Qwen3 Coder 免费模型
    """
    if model_name not in MODEL_CONFIGS:
        available = ", ".join(MODEL_CONFIGS)
        raise ValueError(f"不支持的 model_name: {model_name}。可选值: {available}")

    config = MODEL_CONFIGS[model_name]
    if not config.get("model") or not config.get("base_url"):
        raise ValueError(f"模型 {model_name} 缺少 model 或 base_url 配置，请检查 .env。")
    api_key = os.getenv(config["api_key_env"])
    if not api_key:
        raise ValueError(f"缺少环境变量 {config['api_key_env']}，请先在 .env 中配置。")

    if verbose:
        print(f"创建模型: {model_name} | {config['description']}")

    return init_chat_model(
        model=config["model"],
        model_provider="openai",
        base_url=config["base_url"],
        api_key=api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=temperature,
        max_tokens=max_tokens,
    )


### 1. LangChain 1.0 基础模型消息格式说明与模型调用流程

&emsp;&emsp;在 LangChain 1.0 中，Message（消息）是模型交互的最基本单元。它既代表模型接收到的输入（Input），也代表模型生成的输出（Output）。

&emsp;&emsp;简单来说，每一轮与大模型的对话，都由一条或多条 Message 构成。每个 Message 不仅包含文字内容，还携带描述上下文状态的元信息（metadata），用于保持对话的一致性和可追踪性。通过这种结构化方式，LangChain 可以让模型在多轮交互中理解“谁在说话”“说了什么”“这条信息属于哪一轮对话”。

| 字段                | 说明                                                            | 示例                                                   |
| ----------------- | ------------------------------------------------------------- | ---------------------------------------------------- |
| **Role（角色）**      | 指明消息的类型或来源。常见的有：`system`（系统提示）、`user`（用户输入）、`assistant`（模型回复） | `"role": "user"`                                     |
| **Content（内容）**   | 消息的实际内容，可包含文本、图像、音频、文档等多模态数据                                  | `"content": "你好，请帮我总结以下段落。"`                         |
| **Metadata（元数据）** | 可选字段，存储额外信息，如：消息ID、响应时间、token消耗量、消息标签等                        | `"metadata": {"message_id": "abc123", "tokens": 54}` |


同时，LangChain 在 1.0 中提供了跨模型统一的 Message 标准。无论你使用的是 OpenAI、Anthropic、Gemini 还是本地模型，这一标准都能保持一致的行为。这种统一抽象带来三个好处：其一是，兼容性强：不同模型的消息格式自动对齐。其二是可扩展性高：方便添加多模态内容或自定义字段。其三是可追踪性好：为 LangSmith 等调试工具提供一致的上下文数据结构。

&emsp;&emsp;基本使用方法如下：

In [ ]:
# 后续示例统一复用 `create_model()` 创建的模型；默认是 deepseek-v4-flash，也可以手动选择其它模型
model=create_model()

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

In [ ]:
system_msg = SystemMessage("你叫小智，是一名助人为乐的助手。")
human_msg = HumanMessage("你好，好久不见，请介绍下你自己。")

In [ ]:
messages = [system_msg, human_msg]

In [ ]:
messages

In [ ]:
print("【要点】执行一次基础 invoke：输入是消息列表，输出也是一条 AIMessage。")
response = model.invoke(messages)  

In [ ]:
# 返回的结果也是一条message
response

返回的内容结果解释如下：

| 字段名称                     | 示例值 / 类型                                                                                                   | 说明与作用                                                                       |
| ------------------------ | ---------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------- |
| **content**              | `'你好呀！我是小智，一个乐于助人的智能助手。...'` | **模型生成的主要文本内容**。即对话中 AI 的自然语言回复。                                            |
| **additional_kwargs**    | `{...}`                                                                                                    | **附加参数字段**，通常存放模型服务返回的补充信息。不同服务商返回内容可能不同。                             |
| **response_metadata**    | `{...}`（字典对象）                                                                                              | **响应元数据**，用于记录模型调用的底层信息，如 token 使用量、模型来源、运行 ID 等。详解如下：                     |
| ↳ `token_usage`          | `{'completion_tokens': 60, 'prompt_tokens': 25, 'total_tokens': 85, ...}`                                  | 记录生成所消耗的 **token 统计信息**。具体字段会随模型服务商略有差异。                                  |
| ↳ `model_provider`       | `'openai'`                                                                                                 | 这里使用 OpenAI 兼容协议接入 SiliconFlow，因此 LangChain 侧的 provider 为 `openai`。                  |
| ↳ `model_name`           | `'Qwen/Qwen3-8B'`                                                                                          | 模型的具体名称。                                                                    |
| ↳ `system_fingerprint`   | `...`                                                                                                      | 模型版本或运行实例指纹，用于调试和追踪；不同服务商可能不返回该字段。                                      |
| ↳ `id`                   | `...`                                                                                                      | 当前响应在模型服务或 LangChain 运行中的唯一标识符。                                              |
| ↳ `finish_reason`        | `'stop'`                                                                                                   | 模型生成结束的原因，`stop` 表示自然结束（无截断或超长）。                                            |
| **id**                   | `'lc_run--...'`                                                                                            | LangChain 内部为此消息生成的唯一 **运行 ID**。                                            |
| **usage_metadata**       | `{...}`（字典对象）                                                                                              | LangChain 层的 **统一 token 计量信息**，与 `response_metadata.token_usage` 相呼应。                |
| ↳ `input_tokens`         | `25`                                                                                                       | 输入 token 数量。                                                                |
| ↳ `output_tokens`        | `60`                                                                                                       | 输出 token 数量。                                                                |
| ↳ `total_tokens`         | `85`                                                                                                       | 总 token 数量（输入 + 输出）。                                                        |
| ↳ `input_token_details`  | `{...}`                                                                                                    | 输入部分缓存命中等补充信息。                                                             |
| ↳ `output_token_details` | `{...}`                                                                                                    | 输出部分的补充统计。                                                                  |


据此，我们就能通过拼接消息列表来创建多轮对话或者进行Few-shot提示：

In [ ]:
messages = [
    SystemMessage("你叫小智，是一名助人为乐的助手。"),
    HumanMessage("你好，我叫陈明，好久不见，请介绍下你自己。"),
    AIMessage("你好呀！我是小智，一个乐于助人的智能助手。我的主要功能是回答问题、提供建议、协助解决问题，或者陪你聊天。无论是学习、工作、生活琐事，还是想找点有趣的话题，我都可以帮忙！如果有任何需要，随时告诉我哦～ 😊"),
    HumanMessage("你好，请问你还记得我叫什么名字么？"),
]

In [ ]:
response = model.invoke(messages)  

In [ ]:
response.content

除此之外，如果只需要输入一段简单的文本提示，也可以采用类似第二部分的调用方法，在invoke中直接输入文本，即可自动转化为user message并进行对话：

In [ ]:
question = "你好，请你介绍一下你自己。"

result = model.invoke(question)
print(result.content)

### 2. 基础模型的流式响应和批处理

&emsp;&emsp;在 LangChain 中，大多数模型都支持流式生成（Streaming Generation），即在模型输出完整回答之前，边生成边输出结果。这种机制使得：
- 响应速度更快 — 用户不必等待完整输出；
- 交互体验更流畅 — 尤其在长文本或复杂推理场景下；
- 可实时展示模型思考过程。

&emsp;&emsp;在 LangChain 1.0 中，可以通过调用stream()来进行流式响应，当使用stream()方法后，模型不会一次性返回完整结果，而是返回一个迭代器（iterator），每次迭代都会生成一个 AIMessageChunk，包含部分生成内容，运行结果会实时打印模型输出的 token：

In [ ]:
print("【要点】演示流式响应：模型生成一段，就返回一段，适合聊天界面。")
for chunk in model.stream("你好，好久不见"):
    print(chunk.text, end="|", flush=True)

每个 AIMessageChunk 都可以通过加法 + 操作符拼接。LangChain 内部为此设计了“消息块相加（chunk summation）”机制。

In [ ]:
full = None  # 初始值为空
for chunk in model.stream("你好，好久不见"):
    full = chunk if full is None else full + chunk
    print(full.text)

拼接完成后，可以查看完整内容块：

In [ ]:
print(full.content_blocks)

需要注意的是：
1. 流式输出依赖于整个程序链路都支持“逐块处理”。如果程序中的某个环节必须等待完整输出（如需一次性写入数据库），则无法直接使用 Streaming；
2. angChain 1.0 进一步优化了流式机制，引入 自动流式模式（Auto-streaming）。例如在Agent中，如果整体程序处于 streaming 模式，即便节点中调用 model.invoke()，LangChain 也会自动流式化模型调用。

此外，LangChain 还支持通过 astream_events() 对语义事件进行异步流式监听，适合需要过滤不同事件类型的复杂场景。

&emsp;&emsp;而在使用大模型时，如果需要同时处理多条独立请求（例如多个问题或多段文本），则可以使用 批量调用（Batch） 方法一次性提交这些请求。LangChain 中的 batch() 方法允许你同时发送一组请求，模型会在后台并行处理，然后返回所有结果：

In [ ]:
print("【要点】演示 batch 批处理：一次提交多条独立请求，观察返回列表。")
responses = model.batch([
    "请介绍下你自己。",
    "请问什么是机器学习？",
    "你知道机器学习和深度学习区别么？"
])

for response in responses:
    print(response)

| 特性       | 说明                                                                                |
| -------- | --------------------------------------------------------------------------------- |
| **执行位置** | `batch()` 在客户端（Client-side）并行调用模型，而非调用模型提供商的批量API（如OpenAI或Anthropic自带的batch API）。 |
| **返回结果** | 默认会在所有任务完成后，统一返回完整结果列表。                                                           |
| **并行优势** | 多条独立请求可同时执行，无需等待彼此完成。                                                             |
| **适用场景** | 文档摘要、批量问答、数据预处理、多样本分类等。                                                           |


当然，我们也可以进行流式批处理，也就是每个任务完成后就立即获取结果（而不是等待全部完成），可以使用 batch_as_completed() 方法。

In [ ]:
for response in model.batch_as_completed([
    "请介绍下你自己。",
    "请问什么是机器学习？",
    "你知道机器学习和深度学习区别么？"
]):
    print(response)

注意：结果可能不按原输入顺序返回，因为不同请求完成时间不同，并且每个返回项都包含原始输入的 index 索引，可根据索引重新排序。

而为了更好的控制并发，我们还可以在config参数中设置批处理的并发数，例如
```python
model.batch(
    list_of_inputs,
    config={
        'max_concurrency': 5,  # 最多并行 5 个请求
    }
)

```

更多config参数解释如下：
| 属性名               | 类型      | 说明             |
| ----------------- | ------- | -------------- |
| `max_concurrency` | `int`   | 最大并行执行数        |
| `timeout`         | `float` | 每个请求的最大超时时间（秒） |
| `callbacks`       | `list`  | 触发事件回调，用于日志或监控 |
| `metadata`        | `dict`  | 额外的上下文信息，可用于追踪 |


### 3. 基础模型的结构化输出方法

&emsp;&emsp;LangChain 1.0为所有模型都设置了结构化输出方法，可以让模型的回复严格匹配一份给定的 Schema（模式），便于解析与后续处理。结构化输出能避免“自然语言歧义”，让下游流程（数据库写入、API调用、连环任务）稳、准、可测。

&emsp;&emsp;目前LangChain 1.0支持多种Schema与结构化输出方式：
- Pydantic（字段校验、描述、嵌套结构，功能最丰富）
- TypedDict（轻量类型约束）
- JSON Schema（与前后端/跨语言接口最通用）

其中Pydantic 提供字段校验、默认值、描述信息、多层嵌套等高级能力，是生产场景首选。

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

这里我们使用 Pydantic 的 BaseModel 定义一个严格的数据结构。每个字段都明确了类型（如 str、int、float），并用 Field(..., description="...") 提供语义描述。据此，模型回复时，LangChain 会要求 LLM 的输出必须能填充这些字段。

然后使用 `with_structured_output()` 引导模型进行结构化输出。这里显式设置 `method="function_calling"`，兼容性比默认 `json_schema` 更好，适合 DeepSeek、SiliconFlow 等 OpenAI 兼容模型服务：

In [ ]:
# model=create_model("Qwen/Qwen3-8B")

In [ ]:
# print("【要点】with_structured_output 会让模型按 Pydantic Schema 返回结构化结果。")
# model_with_structure = model.with_structured_output(Movie)
# response = model_with_structure.invoke("Provide details about the movie Inception")
# print(response)

In [ ]:
# response

而如果想要获得模型的完整回复，则可以设置`include_raw=True`

In [ ]:
# model_with_structure = model.with_structured_output(Movie, include_raw=True)
# resp = model_with_structure.invoke("Provide details about the movie Inception")
# resp

### 4.【入门实践】搭建流式响应的多轮问答机器人

&emsp;&emsp;在本小节结束前，我们再通过一个简单的示例，帮大家更加深刻的理解LangChain 1.0中模型调用方法、流式响应以及通过消息队列的拼接来创建多轮聊天对话机器人的流程。

In [ ]:
print("【要点】把消息历史、流式输出和模型调用组合成一个最小聊天机器人。")
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# 1️⃣ 复用前面通过 `create_model()` 初始化好的模型
# 如果单独运行本单元，请先执行本章节开头的模型初始化单元。

# 2️⃣ 初始化系统提示词（System Prompt）
system_message = SystemMessage(
    content="你叫小智，是一名乐于助人的智能助手。请在对话中保持温和、有耐心的语气。"
)

# 3️⃣ 初始化消息历史
messages = [system_message]

import sys

RUN_INTERACTIVE_CHAT = True  # 手动交互时改为 True
print("🔹 输入 exit 退出对话\n")
if not RUN_INTERACTIVE_CHAT:
    print("⏭️ 交互聊天默认不自动执行；需要体验时将 RUN_INTERACTIVE_CHAT 设为 True。")

# 4️⃣ 主循环（支持多轮对话 + 流式输出）
while RUN_INTERACTIVE_CHAT:
    user_input = input("👤 你：")
    if user_input.lower() in {"exit", "quit"}:
        print("🧩 对话结束，再见！")
        break

    # 追加用户消息
    messages.append(HumanMessage(content=user_input))

    # 实时输出模型生成内容
    print("🤖 小智：", end="", flush=True)
    full_reply = ""

    # ✅ LangChain 1.0 标准写法：流式输出
    for chunk in model.stream(messages):
        if chunk.content:
            print(chunk.content, end="", flush=True)
            full_reply += chunk.content

    print("\n" + "-" * 40)  # 分隔线

    # 追加 AI 回复消息
    messages.append(AIMessage(content=full_reply))

    # 保持消息长度（只保留最近50轮）
    messages = messages[-50:]


# 第二章 Langchain1.0全新 Agent 框架介绍

In [ ]:
import os
import httpx
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

# 清理系统代理环境变量，避免本地代理配置影响 API 请求
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy",
]:
    os.environ.pop(k, None)

# 如果你的 .env 就在当前目录，也可以直接使用 load_dotenv(override=True)
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

MODEL_CONFIGS = {
    "deepseek-v4-flash": {
        "model": "deepseek-v4-flash",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的轻量快速模型",
    },
    "deepseek-v4-pro": {
        "model": "deepseek-v4-pro",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的更强能力模型",
    },
    "vllm-qwen122b": {
        "model": os.getenv("vLLM_QWEN122B_MODEL_ID"),
        "base_url": os.getenv("vLLM_QWEN122B_BASE_URL"),
        "api_key_env": "vLLM_QWEN122B_API_KEY",
        "description": "vLLM 部署的 Qwen 122B 模型（默认）",
    },
    "Qwen/Qwen3-8B": {
        "model": "Qwen/Qwen3-8B",
        "base_url": "https://api.siliconflow.cn/v1",
        "api_key_env": "SILICONFLOW_API_KEY",
        "description": "SiliconFlow 上的 Qwen/Qwen3-8B 模型",
    },
    "qwen/qwen3-coder:free": {
        "model": "qwen/qwen3-coder:free",
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
        "description": "OpenRouter 上的 Qwen3 Coder 免费模型",
    },
}

DEFAULT_MODEL_NAME = "vllm-qwen122b"
# 切换到 DeepSeek：create_model("deepseek-v4-flash") 或 create_model("deepseek-v4-pro")


def create_model(
    model_name: str = DEFAULT_MODEL_NAME,
    temperature: float = 0.3,
    max_tokens: int = 10000,
    verbose: bool = True,
):
    """创建课程统一使用的 OpenAI 兼容聊天模型。

    支持五种选择：
    - vllm-qwen122b：从 .env 读取 vLLM Qwen 122B 配置（默认）
    - deepseek-v4-flash：DeepSeek 兼容 OpenAI API 模型服务
    - deepseek-v4-pro：DeepSeek 兼容 OpenAI API 模型服务
    - Qwen/Qwen3-8B：SiliconFlow 上的 Qwen 模型
    - qwen/qwen3-coder:free：OpenRouter 上的 Qwen3 Coder 免费模型
    """
    if model_name not in MODEL_CONFIGS:
        available = ", ".join(MODEL_CONFIGS)
        raise ValueError(f"不支持的 model_name: {model_name}。可选值: {available}")

    config = MODEL_CONFIGS[model_name]
    if not config.get("model") or not config.get("base_url"):
        raise ValueError(f"模型 {model_name} 缺少 model 或 base_url 配置，请检查 .env。")
    api_key = os.getenv(config["api_key_env"])
    if not api_key:
        raise ValueError(f"缺少环境变量 {config['api_key_env']}，请先在 .env 中配置。")

    if verbose:
        print(f"创建模型: {model_name} | {config['description']}")

    return init_chat_model(
        model=config["model"],
        model_provider="openai",
        base_url=config["base_url"],
        api_key=api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=temperature,
        max_tokens=max_tokens,
    )


&emsp;&emsp;在 LangChain 1.0 中，Agent 的核心入口统一为 `create_agent()`。它把模型、工具、系统提示、记忆和中间件组织成一个可运行的 Agent Runtime。学习时可以先把它理解成：**一个由模型负责决策、由工具负责行动、由运行时负责状态和调度的智能体执行框架**。


### 1. API 统一入口


&emsp;&emsp;LangChain 在 1.0 版本中，将所有 Agent 的创建方式统一为了一个入口——`create_agent()`。它取代了旧版本中的 `create_react_agent`、`create_json_agent`、`create_tool_calling_agent` 等多种分支函数，真正让开发者用一行代码即可创建任何类型的智能体。

&emsp;&emsp;在 LangChain 0.x 时代，框架内的 Agent 系统经历了“碎片化”阶段。当时的设计理念是 **“针对场景设计特定 Agent”**——如果你要实现思维链推理（ReAct），就用 `create_react_agent`；如果需要结构化输出，就用 `create_structured_chat_agent`；要工具调用，则用 `create_tool_calling_agent`。这种方式灵活，但也带来了三个明显问题：
1.  **心智负担高**——每种 Agent 都要单独记忆 API 与参数；
2.  **可组合性差**——多个 Agent 之间无法统一调度；
3.  **生态碎片化**——不同模块难以复用或协同演化。

而进入 LangChain 1.0 后，团队做出了彻底重构：将所有 Agent 的创建入口合并为 `create_agent()`，同时在底层通过“中间件机制（Middleware）”和“标准模型接口（invoke / stream）”实现全局统一。这让框架更轻、更稳，也更易于被集成到其他 Agent 平台中。

### 2. `create_agent` 最小调用流程


&emsp;&emsp;`create_agent()` 的最小使用方式很直接：准备一个模型、准备一组工具，再传入系统提示。运行时，Agent 会根据用户问题决定是否调用工具，以及如何把工具返回结果组织成最终回答。


In [ ]:
from dotenv import find_dotenv, load_dotenv
import os
import getpass

# 1. 从 .env 文件加载环境变量
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

# 2. 获取 TAVILY_API_KEY
api_key = os.getenv("TAVILY_API_KEY")

# 3. 如果没获取到，则手动输入
if not api_key:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [ ]:
print("【要点】创建第一个 Agent：模型 + TavilySearch 工具 + system_prompt。")
# 1.导入相关库
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

# 2.导入模型和工具
web_search = TavilySearch(max_results=2, topic="general")
model = create_model()

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [{"role": "user", "content": "请帮我查询2024年诺贝尔物理学奖得主是谁？"}]}
)


In [ ]:
result['messages'][-1].content

### 3. `create_agent` 的 LangGraph 底层架构


&emsp;&emsp;而从底层架构来看，从 LangChain 1.0 开始，官方对框架的整体架构进行了根本性的角色重置：LangGraph 被正式确立为 LangChain 的底层运行框架（Runtime Foundation），而非附属扩展。在 0.x 时代，LangGraph 还被视作 LangChain 的“高层封装”——它是开发者在 LangChain 基础上进一步实现复杂 Agent 编排、状态流转与多节点控制的可选工具；换言之，那时 LangChain 是核心、LangGraph 是外挂。而进入 1.0 之后，情况彻底反转：LangGraph 被下沉为 LangChain 的底层执行引擎，一切新的高级 API（包括统一的 create_agent()、中间件系统、状态管理机制、流式回调与事件调度等）都直接运行在 LangGraph 架构之上。这样一来，LangChain 不再只是一个“Prompt 组织与调用库”，而是一个基于图结构的智能体运行时系统：每个 Agent 的生命周期、每次模型调用、每个工具执行，都是图中一个节点或边的执行过程。LangGraph 提供了稳定的调度、状态传递与事件流控制，而 LangChain 则提供了上层接口与开发体验。也就是说，上述Agent的实际运行逻辑，如果用LangGraph的语言来描述就是：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028154837987.png" alt="image-20251028154837987" style="zoom:50%;" />

这种统一架构并以LangGraph为底层框架带来的优势如下：

1. **接口统一，降低心智负担**

   * 无论是工具型、规划型还是对话型 Agent，都用同一个入口创建。
   * 开发者不再需要区分不同 Agent 的实现逻辑。

2. **跨生态兼容性增强**

   * LangChain 1.0 的 Agent 架构与 LangGraph、LangServe、甚至 OpenAI Responses API 都完全兼容。
   * 这意味着同一个 Agent 可以同时运行在命令行、Web UI、LangGraph 节点或云端推理环境中。

3. **中间件机制的加入**

   * 新版本允许通过装饰器或中间件在 Agent 生命周期中插入逻辑：
     如日志、速率限制、记忆压缩、PII 审查、内容重写等。
   * 这使得 Agent 更像一个“微型操作系统进程”，具备可观察性与可管理性。

4. **可扩展性与定制性更强**

   * 统一 API 底层基于标准化状态流（State Graph），
     这意味着用户可以自由地扩展 Agent 行为而不破坏核心逻辑。

### 4. 与 Agents SDK、ADK 的定位对比


&emsp;&emsp;并且，LangChain 1.0 的 `create_agent()` 与 OpenAI 的 `client.agents.create()`、Google ADK 的 `Agent.create()` 在形式上极为相似：都是 “传入模型 + 工具 + 说明 → 获得一个可交互 Agent 实例”。但 LangChain 的优势在于——它不仅是一个调用层 SDK，而是一个 **完备的 Agent Runtime 框架**。

| 维度               | LangChain 1.0                               | OpenAI Agents SDK           | Google ADK                           |
| ---------------- | ------------------------------------------- | --------------------------- | ------------------------------------ |
| **定位**           | 完整 Agent Runtime（含状态流、中间件、LangGraph）        | 平台级 SDK                     | 研究级 Agent 构建工具                       |
| **创建方式**         | `create_agent(model, tools, system_prompt)` | `client.agents.create(...)` | `Agent.create(tools, model, config)` |
| **状态管理**         | AgentState + Graph                          | 隐式                          | Context Graph                        |
| **中间件支持**        | ✅ 丰富的 before/after hooks                    | ❌（需手动实现）                    | ✅ 部分支持                               |
| **生态完备性**        | 🟢 极高（内置工具库、RAG模块、Serve接口）                  | 中等                          | 中等                                   |

因此我们可以说：**LangChain 1.0 的 Agent 系统兼具 OpenAI SDK 的易用性 + Google ADK 的可扩展性 + 自身生态的完整性。** 它不只是“能用”，更是“能跑得久、能部署、能运维”的生产级智能体框架。

### 5. Agent Runtime 框架


&emsp;&emsp;而从官方的角度进行解释，LangChain 1.0中的Agent系统是一个Agent Runtime框架。在计算机领域中，Runtime 通常指的是一段代码运行所依赖的执行环境，它负责管理：
- 程序的生命周期（从初始化 → 执行 → 结束）；
- 内部状态的维护；
- 与外部资源的交互（输入、输出、网络、文件等）；
- 异常处理、事件调度、中断恢复等底层细节。

而Agent Runtime 框架 = 让智能体能够持续运行、思考、决策、调用工具、保持状态的底层执行系统。它不是一个“函数库”，而是一整套管理 Agent 生命周期与执行流程的框架。当然其中主要得益于LangChain 1.0 Agent框架继承了此前LangGraph的全套开发框架、并在此基础上增加了中间件Middleware，使得Agent开发和运行过程更加可控。

### 6. `create_agent` 的模型接入方式


&emsp;&emsp;在 LangChain 1.0 中，Agent（智能体） 不再只是调用模型的脚本，而是一个能理解任务、推理路径、调用工具并自主完成目标的系统。每个 Agent 都包含一个循环结构（Agent Loop）：模型会根据当前输入进行推理，决定是否调用工具、如何解析观察结果，并不断迭代，直到满足“停止条件”（Stop Condition），也就是模型生成最终答案，或达到设定的迭代次数上限。

LangChain 1.0 提供了全新的 create_agent() 接口，这是一个 可用于生产环境的标准化智能体构造函数。通过它，你可以用极简的方式创建一个完整的、可运行的智能体系统。create_agent 会在内部使用 LangGraph 构建一个基于图结构的 Agent Runtime，它以节点（Node）和边（Edge）的形式定义整个推理流程：
- 节点（Node）：代表执行步骤，例如模型调用节点、工具执行节点或中间件节点；

- 边（Edge）：代表执行流程与状态流转；

- 图（Graph）：定义了 Agent 如何在这些节点之间流动、推理和决策。

换而言之，LangChain 的每个 Agent 都是一张有向图，语言模型是推理引擎，工具是行动节点，而底层则是通过LangGraph进行的编写和实现。


<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028154837987.png" alt="image-20251028154837987" style="zoom:50%;" />

在 create_agent() 的底层运行机制中，最重要的三个组件分别是：

1. 模型（Model）

&emsp;&emsp;模型是 Agent 的“大脑”，负责理解任务与决策推理。它既可以是静态模型（固定不变），也可以是动态模型（根据上下文切换）。

2. 工具（Tools）

&emsp;&emsp;工具是 Agent 的“手脚”。模型可以选择调用不同工具来实现特定子任务，例如：

- web_search() → 进行联网搜索

- calculate() → 计算公式

- database_query() → 查询数据源

3. 中间件（Middleware）

&emsp;&emsp;中间件类似于“神经系统”，它可以在模型调用前后拦截请求、修改参数、或动态调整模型选择逻辑，是 LangChain 1.0 的强大扩展机制。

`create_agent` 支持两种模型使用模式：静态模型（Static Model）与动态模型（Dynamic Model）。其中静态模型是在创建 Agent 时传入一个固定模型实例，并在整个执行过程中保持不变。这种方式简单稳定，适合实验和一般生产场景。例如：

```python
from langchain.agents import create_agent

model = create_model()

agent = create_agent(
    model=model,
    tools=tools,
)
```

本课程后续统一使用显式实例化的模型对象，也就是先通过 `create_model()` 创建模型，再传入 `create_agent`。这种写法便于统一配置 `base_url`、`api_key`、`temperature`、`max_tokens` 等参数，也方便在 `deepseek-v4-flash`、`deepseek-v4-pro`、`Qwen/Qwen3-8B` 和 `qwen/qwen3-coder:free` 之间切换。

> 完整静态模型字符串映射详见：https://reference.langchain.com/python/langchain/models/?_gl=1*125pi5n*_gcl_au*MTI5ODI4NTA5Ny4xNzYxMjA1ODI2*_ga*MTc2OTg3MTQ2OS4xNzYxMzc5MDEw*_ga_47WX3HKKY2*czE3NjE2MzMwNDQkbzYkZzEkdDE3NjE2MzczMzUkajYwJGwwJGgw#langchain.chat_models.init_chat_model(model_provider)
> <center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028164512688.png" alt="image-20251028164512688" style="zoom:50%;" />

另外，在静态模型调用过程中，也可以在创建模型的环节设置更多参数，例如：

```python
from langchain.agents import create_agent

model = create_model(
    temperature=0.1,
    max_tokens=1000,
)

agent = create_agent(model=model, tools=tools)
```

而除此之外，LangChain 1.0 的 `create_agent` API还提供了动态模型的方式，这是一种更高级的机制，它允许 Agent 在运行时根据任务上下文自动切换模型，实现智能的模型路由与成本优化。例如对话初期使用轻量模型（如 gpt-4o-mini）以节省成本；当上下文变复杂或任务难度提升时，自动切换到更强模型（如 gpt-4o）。而此时就需要使用到中间件的@wrap_model_call装饰器，这部分内容我们将放在下一小节进行介绍。

# 第三章 工具：内置工具、自定义工具与 MCP

&emsp;&emsp;工具是 Agent 从“会对话”走向“能做事”的关键。LangChain 中常见的工具接入方式可以分为三类：

| 工具类型 | 适用场景 | 本章示例 |
| --- | --- | --- |
| 内置或生态集成工具 | 已经有成熟第三方集成时，优先直接使用 | `TavilySearch` 网络搜索 |
| 自定义 Python 工具 | 业务逻辑在本地函数中，适合快速封装 | `get_weather` 天气查询函数 |
| MCP 工具服务 | 工具需要跨客户端复用，或希望以独立服务方式部署 | `OpenWeatherMCPServer`、`SecureFileMCPServer` |

&emsp;&emsp;三类工具最终都会变成 LangChain Agent 可调用的 Tool，因此接入 `create_agent` 时形式基本一致。


### 1. 接入生态工具：TavilySearch 网络搜索


&emsp;&emsp;如果只是和模型对话，可以直接使用 `model.invoke()`。一旦希望模型查询外部信息、读取文件或调用业务系统，就需要把这些能力包装成工具，并交给 `create_agent()` 管理。


&emsp;&emsp;先从最常见的生态集成工具开始。LangChain 生态中已经集成了很多常用工具，开发者可以直接复用这些工具完成搜索、浏览器自动化、数据库查询、RAG 检索和文件处理等任务。


- LangChain内置工具列表：https://python.langchain.com/docs/integrations/tools/

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/202506122022882.png" alt="image-20250612202253781" style="zoom:50%;" />

其中典型的工具如下：

| 领域            | 工具名称                                             | 功能说明                             |
| ------------- | ------------------------------------------------ | -------------------------------- |
| **搜索引擎与信息检索** | 🔹 **DuckDuckGo Search**                         | 免费搜索引擎，返回网页标题、摘要与链接，常用于联网问答。     |
|               | 🔹 **Brave Search API**                          | 提供隐私友好的网页搜索结果，可配置输出深度与过滤。        |
|               | 🔹 **Bing Search Tool**                          | 微软 Bing 搜索接口，支持精确检索与多语言查询。       |
| **网页浏览与内容提取** | 🔹 **Playwright Browser Toolkit**                | 提供浏览器自动化访问、网页截图与文本抓取功能。          |
|               | 🔹 **Selenium Toolkit**                          | 类似于 Playwright，但偏向传统网页 DOM 操控。   |
| **生产力工具集成**   | 🔹 **Slack Toolkit**                             | 允许 Agent 在 Slack 中发送、读取或总结消息。    |
|               | 🔹 **Jira Toolkit**                              | 用于查询、创建或更新 Jira 项目工单。            |
|               | 🔹 **Google Drive / Docs Toolkit**               | 操作 Google 文档、表格与文件搜索。            |
| **数据库与数据操作**  | 🔹 **SQLDatabase Toolkit**                       | 连接 SQL 数据库，自动生成并执行 SQL 查询。       |
|               | 🔹 **Faiss / Chroma / Milvus VectorStore Tools** | 进行向量相似度搜索，用于 RAG 检索。             |
| **代码与终端执行**   | 🔹 **Python REPL Tool**                          | 执行 Python 代码片段并返回结果（沙盒环境）。       |
|               | 🔹 **Shell Tool (Bash)**                         | 允许执行 Shell 命令（需安全限制）。            |
|               | 🔹 **Requests Wrapper Tool**                     | 发起 HTTP 请求、API 调用，用于外部数据交互。      |
| **多模态工具**     | 🔹 **DALL-E Tool**                               | 生成图像，可结合文本描述创作内容。                |
|               | 🔹 **SerpAPI Image Search**                      | 图片搜索引擎，可返回图片 URL 与元信息。           |
|               | 🔹 **OpenAI Whisper Tool**                       | 音频转录工具，将语音文件转换为文本。               |
| **知识问答与嵌入检索** | 🔹 **VectorStoreRetrieverTool**                  | 基于嵌入模型检索文档片段，用于 RAG 问答。          |
|               | 🔹 **MultiQueryRetrieverTool**                   | 生成多种查询变体以提升检索覆盖率。                |
| **云与AI服务**    | 🔹 **OpenAI Functions / Azure Functions Tool**   | 让 Agent 调用 OpenAI 或 Azure 定义的函数。 |
|               | 🔹 **Anthropic ToolKit**                         | 集成 Claude 模型工具调用能力。              |
| **文件系统与数据处理** | 🔹 **CSV Toolkit**                               | 读取、过滤与分析 CSV 文件数据。               |
|               | 🔹 **Pandas DataFrame Agent Tool**               | 直接在 DataFrame 上执行查询与统计任务。        |


下面以网络搜索工具为例，演示如何把一个现成工具接入 `create_agent()`。


In [ ]:
# !pip install -qU langchain-tavily -i https://pypi.tuna.tsinghua.edu.cn/simple

本课程使用独立的 `langchain-tavily` 集成包接入 Tavily Search。Tavily 是面向 AI Agent 的实时搜索工具，可以用于网络搜索和信息检索。这里需要先在 Tavily 官网注册并获得 API Key（每月有免费额度）：https://www.tavily.com/


<center>

<img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028165726979.png" alt="image-20251028165726979" style="zoom:50%;" />

然后将 API Key 写到本地 `.env` 文件中的 `TAVILY_API_KEY` 变量中。`TavilySearch` 会默认从环境变量读取该密钥，因此只要前面已经通过 `load_dotenv()` 加载 `.env`，这里就可以直接实例化工具。


In [ ]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [ ]:
print("【要点】实例化 TavilySearch，先把搜索工具当普通工具直接验证。")
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=2,
    topic="general",
)

In [ ]:
web_search

这是一个高度封装的网络搜索工具。按照官方文档推荐，直接调用时传入包含 `query` 字段的字典：


In [ ]:
print("【要点】直接调用 TavilySearch，确认工具输入格式是包含 query 的字典。")
web_search.invoke({"query": "请问 2024 年诺贝尔物理学奖得主是谁？"})

验证工具可以直接调用后，也可以把它交给 `create_agent()`，让 Agent 在需要时自行决定是否搜索。


In [ ]:
print("【要点】把 TavilySearch 交给 Agent，让模型自行决定何时搜索。")
# 1. 导入相关库
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

# 2. 导入模型和工具
web_search = TavilySearch(max_results=2, topic="general")
model = create_model()

# 3. 创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4. 运行Agent获得结果
result = agent.invoke(
    {"messages": [{"role": "user", "content": "请帮我查询2024年诺贝尔物理学奖得主是谁？"}]}
)


In [ ]:
result['messages'][-1].content

而如果仔细观察本次运行过程，本次工具调用仍然是一次典型的Function Calling执行流程，包括用户首次发起消息在内，总共创建了4条消息，分别是user message、function call message、function response message以及final response：


In [ ]:
result['messages']

In [ ]:
len(result['messages'])

In [ ]:
result['messages'][0]

In [ ]:
result['messages'][1]

In [ ]:
result['messages'][2]

In [ ]:
result['messages'][3]

&emsp;&emsp;一次完整的`Function Calling`执行流程如下：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/202412191720637.png" alt="202412191720637" style="zoom:50%;" />


### 2. 接入自定义工具：把 Python 函数交给 Agent


&emsp;&emsp;能调用外部工具，是大模型从“聊天模型”走向 Agent 的关键。大模型本身不会直接访问天气 API、数据库或文件系统，它需要通过工具调用协议表达“我要调用哪个函数、传什么参数”。运行时再由程序执行函数，并把结果返回给模型继续推理。


<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250318202017508.png" alt="image-20250318202017508" style="zoom:35%;" />

&emsp;&emsp;例如，当我们要查询当前天气时，让大模型调用外部工具的Function Calling的过程就如图所示：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20250318202029130.png" alt="image-20250318202029130" style="zoom:35%;" />


&emsp;&emsp;因此，接入自定义工具的核心，是先写一个稳定、边界清晰的 Python 函数。下面用 OpenWeather API 查询实时天气作为例子：


> 这里需要先登录 OpenWeather 官网获取 API Key：https://home.openweathermap.org/ ，然后将其写入 `.env` 中的 `OPENWEATHER_API_KEY` 变量中。

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv 
load_dotenv(find_dotenv(usecwd=True), override=True)

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

In [ ]:
import requests, json
from requests import RequestException
import time


In [ ]:
print("【要点】先写普通 Python 天气函数，这是自定义工具的业务逻辑本体。")

def _fetch_weather_json(loc: str, retries: int = 3, timeout: float = 8.0) -> str:
    """调用 OpenWeather，并把成功或失败都转换成 JSON 字符串返回。"""
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": loc,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn",
    }

    last_error_detail = "未知错误"
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return json.dumps(response.json(), ensure_ascii=False)
        except requests.HTTPError as exc:
            status_code = exc.response.status_code if exc.response is not None else "未知"
            last_error_detail = f"OpenWeather HTTP {status_code}"
            # 不打印 exc 的完整 URL，避免日志泄露 appid。
            print(f"天气接口调用失败，第 {attempt}/{retries} 次：HTTP {status_code}")
            # 4xx（除 429）通常是参数或 Key 问题，继续重试没有意义。
            if status_code != 429 and 400 <= status_code < 500:
                break
        except RequestException as exc:
            last_error_detail = f"{type(exc).__name__}: 网络连接失败"
            print(f"天气接口调用失败，第 {attempt}/{retries} 次：{last_error_detail}")

        if attempt < retries:
            # 指数退避，避免在上游 502/429 时瞬间重试。
            time.sleep(2 ** (attempt - 1))

    return json.dumps(
        {
            "ok": False,
            "location": loc,
            "error": "OpenWeather request failed",
            "detail": last_error_detail,
            "suggestion": "网络或远端接口临时异常；教学中可稍后重试，或改用后文 MCP 天气服务示例。",
        },
        ensure_ascii=False,
    )


def get_weather(loc):
    """
    查询即时天气函数。
    :param loc: 城市英文名，例如 Beijing、Shanghai。
    :return: OpenWeather API 查询结果；如果网络失败，也返回结构化 JSON 错误信息。
    """
    return _fetch_weather_json(loc)


In [ ]:
get_weather("Beijing")

紧接着将其封装为LangChain能够识别的外部函数：

In [ ]:
from langchain.tools import tool

In [ ]:
print("【要点】用 @tool 把普通函数包装成 LangChain Tool，函数签名和 docstring 会变成工具说明。")
@tool
def get_weather(loc: str) -> str:
    """
    查询即时天气函数。
    :param loc: 城市英文名，例如 Beijing、Shanghai。
    :return: OpenWeather API 查询结果；如果网络失败，也返回结构化 JSON 错误信息。
    """
    return _fetch_weather_json(loc)


In [ ]:
print(get_weather.name)
print(get_weather.description)
print(get_weather.args)

然后即可直接带入到create_agent中，创建一个简易的天气查询Agent：

In [ ]:
print("【要点】创建天气 Agent：模型根据用户问题决定是否调用 get_weather。")
# 1. 导入相关库
from langchain.agents import create_agent

# 2. 导入模型
model = create_model()

# 3. 创建Agent
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=(
        "你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
        "查询多个城市时，即使某个城市返回结构化错误，也要继续查询其余城市。"
        "最后必须调用 write_file，将每个城市的成功数据或失败原因一起写入文件。"
    )
)

# 4. 运行Agent获得结果
result = agent.invoke(
    {"messages": [{"role": "user", "content": "请问北京今天天气如何？"}]}
)


In [ ]:
result['messages'][-1].content

而同样，这也是一个典型的Function Calling流程。


In [ ]:
len(result['messages'])

### 3. 接入 MCP 工具：将外部服务标准化为 Agent 工具

&emsp;&emsp;前面我们已经学习了两类工具接入方式：一种是 LangChain 生态集成工具，例如 `TavilySearch`；另一种是用 `@tool` 把本地 Python 函数包装成工具。这两种方式都能工作，但工具和 Agent 代码通常耦合在同一个 Python 项目里。

&emsp;&emsp;MCP（Model Context Protocol，模型上下文协议）解决的是另一类问题：把外部能力封装成独立服务，由 MCP 客户端按协议发现工具、读取工具 schema、发起工具调用。这样，一个工具服务可以被多个客户端复用，例如 Cline、Claude Desktop、Cherry Studio、Codex 或 LangChain Agent。


#### 3.1 MCP 与 Function Calling 的关系

&emsp;&emsp;从执行本质上看，MCP 并没有改变大模型调用工具的基本逻辑。大模型仍然需要先决定“要不要调用工具、调用哪个工具、传入什么参数”，这仍然是 Function Calling / Tool Calling 的范畴。

&emsp;&emsp;MCP 主要标准化的是工具服务层：

| 层次 | 关注点 | 说明 |
| --- | --- | --- |
| Function Calling | 模型如何表达工具调用意图 | 由模型输出工具名和参数 |
| LangChain Tool | Python 侧如何把函数交给 Agent | 通常是本地函数或集成包 |
| MCP Server | 工具如何作为独立服务暴露 | 工具 schema、调用、传输协议标准化 |
| MCP Client | 如何连接 MCP Server 并加载工具 | LangChain 中可用 `MultiServerMCPClient` |

&emsp;&emsp;因此，可以把 MCP 理解为“更标准、更可复用的工具服务协议”。



#### 3.2 FastMCP：快速开发 MCP Server

&emsp;&emsp;`FastMCP` 是 MCP Python SDK 中用于快速创建 MCP Server 的高层封装。它的开发体验很接近 FastAPI：先创建服务实例，再用装饰器把普通 Python 函数注册成 MCP 工具。

典型结构如下：

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoMCPServer", port=16681)

@mcp.tool()
def ping() -> dict:
    return {"ok": True}

if __name__ == "__main__":
    mcp.run(transport="streamable-http")
```

&emsp;&emsp;本课程中的两个 MCP 服务都采用 `streamable-http` 传输方式，因此 LangChain 端可以通过 HTTP URL 连接它们。



#### 3.3 本课程提供的两个 MCP 服务

本目录下已经准备了两个相对完整的 MCP 服务：

| 服务 | 入口脚本 | 默认地址 | 工具能力 |
| --- | --- | --- | --- |
| `OpenWeatherMCPServer` | `mcp/01_weather-mcp-server/openweather_mcp_server.py` | `http://127.0.0.1:16681/mcp` | 天气服务健康检查、OpenWeather 配置检查、实时天气查询 |
| `SecureFileMCPServer` | `mcp/02_file-mcp-server/secure_file_mcp_server.py` | `http://127.0.0.1:16682/mcp` | 文件服务健康检查、列目录、读文件、文件信息、同目录重命名、查看与恢复重命名历史 |

&emsp;&emsp;其中 `SecureFileMCPServer` 默认带有目录边界限制；下面的示例会把允许访问根目录设置为当前课程目录，确保示例可以直接列出当前目录中的文件。



In [ ]:
print("【要点】启动两个本地 MCP Server：天气服务和受限文件服务。")
import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
WEATHER_MCP_SCRIPT = NOTEBOOK_DIR / "mcp/01_weather-mcp-server" / "openweather_mcp_server.py"
FILE_MCP_SCRIPT = NOTEBOOK_DIR / "mcp/02_file-mcp-server" / "secure_file_mcp_server.py"

MCP_PROCESSES = {}


def is_port_open(port: int, host: str = "127.0.0.1", timeout: float = 0.5) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def start_mcp_server(name: str, script_path: Path, port: int, extra_env: dict | None = None):
    """启动一个本地 streamable-http MCP Server；如果端口已打开，则认为服务已存在。"""
    if is_port_open(port):
        return {"name": name, "port": port, "status": "already_running"}

    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)

    process = subprocess.Popen(
        [sys.executable, str(script_path)],
        cwd=str(script_path.parent),
        env=env,
        # 不使用未持续消费的 PIPE，避免 MCP 日志写满缓冲区后阻塞。
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        text=True,
    )

    for _ in range(40):
        if is_port_open(port):
            MCP_PROCESSES[name] = process
            return {"name": name, "port": port, "status": "started", "pid": process.pid}
        if process.poll() is not None:
            stderr = process.stderr.read() if process.stderr else ""
            raise RuntimeError(f"{name} 启动失败：{stderr}")
        time.sleep(0.25)

    process.terminate()
    raise TimeoutError(f"{name} 在端口 {port} 上启动超时。")


weather_status = start_mcp_server(
    name="weather",
    script_path=WEATHER_MCP_SCRIPT,
    port=16681,
    extra_env={"WEATHER_MCP_PORT": "16681"},
)

file_status = start_mcp_server(
    name="file",
    script_path=FILE_MCP_SCRIPT,
    port=16682,
    extra_env={"FILE_MCP_MAX_READ_BYTES": "200000"},
)

weather_status, file_status


#### 3.4 在 LangChain 中加载 MCP 工具

&emsp;&emsp;LangChain 通过 `langchain-mcp-adapters` 将 MCP Server 暴露的工具转换成 LangChain Tool。这里使用 `MultiServerMCPClient` 同时连接天气服务和文件服务。

&emsp;&emsp;注意：两个 MCP 服务都提供了 `ping` 工具，为避免重名，下面设置 `tool_name_prefix=True`，这样工具名会变成 `weather_ping`、`file_ping` 这类带服务名前缀的形式。



In [ ]:
print("【要点】用 MultiServerMCPClient 连接 MCP Server，并把远程工具加载成 LangChain Tool。")
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "weather": {
            "url": "http://127.0.0.1:16681/mcp",
            "transport": "http",
        },
        "file": {
            "url": "http://127.0.0.1:16682/mcp",
            "transport": "http",
        },
    },
    tool_name_prefix=True,
)

mcp_tools = await mcp_client.get_tools()
[(tool.name, tool.description[:80]) for tool in mcp_tools]


#### 3.5 先直接调用 MCP 工具验证链路

&emsp;&emsp;在交给 Agent 之前，建议先直接调用 MCP 工具验证链路是否正常。这一步不依赖大模型，只检查 MCP Server、MCP Client 和工具 schema 是否能正常工作。



In [ ]:
print("【要点】先直接调用 MCP 工具做链路验证，排除模型因素。")
import json


def parse_mcp_tool_result(result):
    """langchain-mcp-adapters 返回 content blocks；这里把其中的 JSON text 解析成 dict。"""
    if isinstance(result, list) and result and isinstance(result[0], dict):
        text = result[0].get("text", "")
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {"raw_text": text}
    return result


tool_by_name = {tool.name: tool for tool in mcp_tools}

weather_ping_result = parse_mcp_tool_result(await tool_by_name["weather_ping"].ainvoke({}))
file_roots_result = parse_mcp_tool_result(await tool_by_name["file_list_allowed_roots"].ainvoke({}))

weather_ping_result, file_roots_result


In [ ]:
# 调用天气配置检查：即使没有 OpenWeather API Key，也会返回结构化配置状态，不会发起外部天气请求。
parse_mcp_tool_result(await tool_by_name["weather_check_config"].ainvoke({}))


In [ ]:
# 调用安全文件 MCP：列出当前课程目录下的文件和子目录。
parse_mcp_tool_result(await tool_by_name["file_list_files"].ainvoke({"directory": "."}))


#### 3.6 使用 create_agent 接入 MCP 工具

&emsp;&emsp;当 MCP 工具被加载成 LangChain Tool 后，接入 `create_agent` 的方式和普通工具完全一致：把 `mcp_tools` 传入 `tools` 参数即可。

&emsp;&emsp;下面这个示例会要求 Agent 调用两个 MCP 工具：先检查天气 MCP 配置，再列出文件 MCP 的允许根目录。该单元需要前面已经初始化好 `create_model()`，并且 `.env` 中配置了所选模型对应的 API Key，例如 `DEEPSEEK_API_KEY`、`SILICONFLOW_API_KEY` 或 `OPENROUTER_API_KEY`。



In [ ]:
print("【要点】把 MCP 工具交给 create_agent，MCP 工具和普通 Tool 的接入方式一致。")
from langchain.agents import create_agent

mcp_agent_model = create_model(temperature=0.1, max_tokens=1500)

mcp_agent = create_agent(
    model=mcp_agent_model,
    tools=mcp_tools,
    system_prompt=(
        "你是一名会使用 MCP 工具的助手。"
        "当用户要求检查 MCP 状态时，必须优先调用可用工具，而不是只凭常识回答。"
    ),
)

mcp_agent_result = await mcp_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "请调用 MCP 工具完成两件事：1）检查天气 MCP 配置是否可用；2）列出文件 MCP 当前允许访问的根目录。最后用中文总结结果。",
            }
        ]
    }
)

mcp_agent_result["messages"][-1].content


#### 3.7 关闭本 Notebook 启动的 MCP 服务

&emsp;&emsp;如果服务是由上面的 `start_mcp_server()` 单元启动的，可以在课程结束后运行下面的单元释放子进程。如果端口原本就已有服务在运行，本单元不会关闭那些外部服务。



In [ ]:
for name, process in list(MCP_PROCESSES.items()):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
    print(f"{name} MCP server stopped")

MCP_PROCESSES.clear()


### 4. ReAct：Agent 的思考-行动循环

&emsp;&emsp;前面我们已经知道：工具让 Agent 具备了“行动能力”。但真正让 Agent 区别于一次性 Function Calling 的，是它可以在“模型判断”和“工具执行”之间反复循环，直到得到足够信息并生成最终答案。这个循环通常被称为 **ReAct**。

&emsp;&emsp;ReAct 可以拆成两个词：

| 组成 | 含义 | 在 Agent 中对应什么 |
| --- | --- | --- |
| Reason | 推理、判断、规划下一步 | 模型根据用户问题和历史消息决定下一步动作 |
| Act | 行动、调用外部工具 | Agent 调用搜索、天气、文件、数据库等工具 |

&emsp;&emsp;一个典型 ReAct 循环如下：

1. 用户提出任务。
2. 模型读取当前消息和可用工具列表。
3. 模型判断是否已经能回答。
4. 如果不能回答，模型选择一个工具并生成工具参数。
5. Agent 执行工具，得到 Observation（观察结果）。
6. 模型把 Observation 放回上下文，继续判断下一步。
7. 当信息足够时，模型停止调用工具并输出最终答案。

&emsp;&emsp;因此，ReAct 的重点不是“模型一定要把思考过程写出来”，而是 **Agent 运行时存在一个可迭代的 Reason → Act → Observe → Reason 循环**。在 LangChain 1.0 中，这个循环由 `create_agent` 底层的 LangGraph Runtime 管理。


<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028154837987.png" alt="image-20251028154837987" style="zoom:50%;" />

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028171815049.png" alt="image-20251028171815049" style="zoom:50%;" />

#### 4.1 并行调用：一个问题触发多个工具


In [ ]:
print("【要点】观察并行工具调用：一个问题里多个城市天气彼此独立，可在同一轮触发多个工具。")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "请问今天杭州和南昌哪里更冷？"}]}
)

In [ ]:
result['messages'][-1].content

&emsp;&emsp;这个例子里，用户一次性问了多个城市的天气。每个城市的天气查询彼此独立，因此 Agent 可以在同一轮决策中触发多个工具调用。教学上可以把它理解为：**同一阶段需要多份互不依赖的信息时，Agent 可以并行调用工具**。


In [ ]:
result

#### 4.2 多步调用：后一个动作依赖前一个工具结果


In [ ]:
import os
from datetime import datetime

In [ ]:
@tool
def write_file(content: str) -> str:
    """
    将指定内容写入本地文件。
    :param content: 必要参数，字符串类型，用于表示需要写入文档的具体内容。
    :return: 写入结果提示信息。
    """
    try:
        # ✅ 始终先定义文件名（防止未绑定变量）
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"output_{timestamp}.txt"

        # 写入文件
        with open(filename, "w", encoding="utf-8") as f:
            f.write(content)

        abs_path = os.path.abspath(filename)
        return f"✅ 已成功写入本地文件：{abs_path}"

    except Exception as e:
        return f"❌ 文件写入失败：{str(e)}"

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_weather, write_file],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

In [ ]:
print("【要点】观察多步 ReAct：先查天气，再根据观察结果写文件。")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "请帮我查询天津、石家庄、上海等地天气，并写入本地文件。"}]}
)

In [ ]:
result['messages'][-1].content

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028172757736.png" alt="image-20251028172757736" style="zoom:50%;" />

&emsp;&emsp;这个例子更能体现 ReAct 循环：Agent 先调用天气工具拿到观察结果，再根据观察结果决定调用文件写入工具。也就是说，第二个工具调用依赖第一个工具返回的信息，这就是典型的多步 ReAct。


In [ ]:
len(result['messages'])

#### 4.3 ReAct 循环什么时候停止？

&emsp;&emsp;ReAct 循环不会无限执行。通常有三类停止条件：

| 停止条件 | 说明 |
| --- | --- |
| 信息足够 | 模型认为已经可以回答用户问题 |
| 工具失败或无效 | 工具调用报错，或多次重试后仍无法获得有效结果 |
| 达到运行限制 | 例如最大迭代次数、超时、人工中断或中间件拦截 |

&emsp;&emsp;如果出现工具调用报错，Agent 通常会尝试根据错误信息修正参数并重试。关于工具错误如何被改写、拦截或要求人工审批，会在后面的“中间件机制”章节继续展开。


In [ ]:
result['messages']

# 第四章 记忆管理

&emsp;&emsp;本章承接工具调用，集中学习 `create_agent` 的记忆管理方式，包括多轮对话状态、线程隔离和上下文持久化等内容。


### 1. 用 `checkpointer` 保存多轮对话状态


&emsp;&emsp;实际开发 Agent 时，记忆管理非常关键。没有记忆时，每次调用都像新对话；加入 `checkpointer` 后，Agent 可以按 `thread_id` 保存和恢复同一条会话线程中的历史消息。


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
tools = [get_weather]

In [ ]:
print("【要点】给 Agent 加 checkpointer，开始按 thread_id 保存对话状态。")
agent = create_agent(model=model, 
                     tools=tools,
                     checkpointer=checkpointer)

In [ ]:
config = {
    "configurable": {
        "thread_id": "1"  
    }
}

In [ ]:
print("【要点】第一次使用 thread_id=1 对话，记忆会写入当前线程。")
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我叫陈明，好久不见！"}]},
    config
)

In [ ]:
response['messages']

In [ ]:
response['messages'][-1].content

&emsp;&emsp;此时记忆已经保存在当前 Agent 的 checkpointer 中，并且归属于 `thread_id="1"` 这条线程：


In [ ]:
latest = agent.get_state(config)

In [ ]:
latest

&emsp;&emsp;再次调用时，只要继续传入同一个 `thread_id`，Agent 就可以读取此前对话状态：


In [ ]:
print("【要点】复用同一个 thread_id，Agent 应该能记住上一轮用户信息。")
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，请问你还记得我叫什么名字么？"}]},
    config
)

In [ ]:
response['messages']

In [ ]:
response['messages'][-1].content

In [ ]:
latest = agent.get_state(config)
latest

&emsp;&emsp;如果换一个新的 `thread_id`，就会开启一条独立会话，不会自动继承上一条线程的记忆：


In [ ]:
config2 = {
    "configurable": {
        "thread_id": "2"  
    }
}

In [ ]:
print("【要点】切换到新的 thread_id，会开启一条独立会话，不共享上一条线程记忆。")
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，你还记得我叫什么名字么？"}]},
    config2
)

In [ ]:
response2['messages']

# 第五章 中间件机制

&emsp;&emsp;本章整合原 `Part 3.LangChain Agent中间件入门介绍.ipynb`，集中介绍 LangChain 1.0 中间件的核心能力，包括动态模型选择、消息压缩和 Human-in-the-loop。Tavily 工具已统一使用新版 `langchain-tavily` 的 `TavilySearch` 写法。


In [ ]:
import os
import httpx
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

# 清理系统代理环境变量，避免本地代理配置影响 API 请求
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy",
]:
    os.environ.pop(k, None)

# 如果你的 .env 就在当前目录，也可以直接使用 load_dotenv(override=True)
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

MODEL_CONFIGS = {
    "deepseek-v4-flash": {
        "model": "deepseek-v4-flash",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的轻量快速模型",
    },
    "deepseek-v4-pro": {
        "model": "deepseek-v4-pro",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的更强能力模型",
    },
    "vllm-qwen122b": {
        "model": os.getenv("vLLM_QWEN122B_MODEL_ID"),
        "base_url": os.getenv("vLLM_QWEN122B_BASE_URL"),
        "api_key_env": "vLLM_QWEN122B_API_KEY",
        "description": "vLLM 部署的 Qwen 122B 模型（默认）",
    },
    "Qwen/Qwen3-8B": {
        "model": "Qwen/Qwen3-8B",
        "base_url": "https://api.siliconflow.cn/v1",
        "api_key_env": "SILICONFLOW_API_KEY",
        "description": "SiliconFlow 上的 Qwen/Qwen3-8B 模型",
    },
    "qwen/qwen3-coder:free": {
        "model": "qwen/qwen3-coder:free",
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
        "description": "OpenRouter 上的 Qwen3 Coder 免费模型",
    },
}

DEFAULT_MODEL_NAME = "vllm-qwen122b"
# 切换到 DeepSeek：create_model("deepseek-v4-flash") 或 create_model("deepseek-v4-pro")


def create_model(
    model_name: str = DEFAULT_MODEL_NAME,
    temperature: float = 0.3,
    max_tokens: int = 10000,
    verbose: bool = True,
):
    """创建课程统一使用的 OpenAI 兼容聊天模型。

    支持五种选择：
    - vllm-qwen122b：从 .env 读取 vLLM Qwen 122B 配置（默认）
    - deepseek-v4-flash：DeepSeek 兼容 OpenAI API 模型服务
    - deepseek-v4-pro：DeepSeek 兼容 OpenAI API 模型服务
    - Qwen/Qwen3-8B：SiliconFlow 上的 Qwen 模型
    - qwen/qwen3-coder:free：OpenRouter 上的 Qwen3 Coder 免费模型
    """
    if model_name not in MODEL_CONFIGS:
        available = ", ".join(MODEL_CONFIGS)
        raise ValueError(f"不支持的 model_name: {model_name}。可选值: {available}")

    config = MODEL_CONFIGS[model_name]
    if not config.get("model") or not config.get("base_url"):
        raise ValueError(f"模型 {model_name} 缺少 model 或 base_url 配置，请检查 .env。")
    api_key = os.getenv(config["api_key_env"])
    if not api_key:
        raise ValueError(f"缺少环境变量 {config['api_key_env']}，请先在 .env 中配置。")

    if verbose:
        print(f"创建模型: {model_name} | {config['description']}")

    return init_chat_model(
        model=config["model"],
        model_provider="openai",
        base_url=config["base_url"],
        api_key=api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=temperature,
        max_tokens=max_tokens,
    )


### 1. LangChain 1.0 中间件核心功能介绍

&emsp;&emsp;中间件 Middleware 是 LangChain 1.0 Agent Runtime 的重要扩展机制。借助中间件，开发者可以在模型调用前后、工具调用前后、状态更新等关键位置插入控制逻辑。简单来说，原本固定的 ReAct 循环：


<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/core_agent_loop.png" alt="核心代理循环图" style="zoom:100%;" />

变为这种：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/middleware_final.png" alt="中间件流程图" style="zoom:100%;" />

&emsp;&emsp;会变成一个可以被钩子扩展的运行时流程。通过这些钩子，我们可以在不重写 Agent 主循环的情况下，加入日志、限流、动态模型选择、上下文压缩、安全审核和人工审批等工程逻辑。LangChain 1.0 中间件常见能力如下：


| 功能类别                           | 功能说明                                               | 常用钩子                                           |
| ------------------------------ | -------------------------------------------------- | ---------------------------------------------- |
| **监控 / 日志 /指标**                | 跟踪 Agent 的行为、模型调用次数、token 使用量、工具调用频率等。             | `before_model`、`after_model`、`wrap_model_call` |
| **修改输入 / 上下文工程**               | 在模型调用前修改消息历史、添补系统提示、裁剪上下文、替换模型参数等。                 | `before_model`、`modify_model_request`          |
| **动态模型 /工具路由**                 | 根据当前状态决定使用哪个模型、启用哪些工具，动态调整调用策略。                    | `modify_model_request`、`wrap_model_call`       |
| **控制流程 / 限流 /重试 /降级**          | 对模型或工具调用加上重试机制、超时、降级方案、跳转结束、提前中断。                  | `wrap_model_call`、`before_model`（跳转）           |
| **合规 / 审核（Guardrails / HITL）** | 在调用工具前或后插入人工审核、敏感内容检查、PII检测、安全过滤。                  | `after_model`（或 `wrap_tool_call`）              |
| **摘要 / 上下文裁剪**                 | 当对话变长或token靠近限制时，自动摘要历史、压缩上下文以维持性能。                | `before_model`                                 |
| **工具选择与调用管理**                  | 在模型调度工具前决定哪些工具可用、工具是否被选中、限制调用频率。                    | `modify_model_request`、`wrap_tool_call`        |
| **结构化输出 / 响应格式控制**             | 在请求模型之前或之后规范输出格式、校验结构化数据。                          | `modify_model_request`、`after_model`           |
| **状态扩展**                       | 自定义 Agent 的状态 schema（如调用次数、用户偏好、认证状态），中间件可以读写这些状态。 | `before_model`、`after_model`                   |


> 钩子是框架或系统在某些关键执行点暴露的扩展接口。开发者可以“挂上”自己的逻辑，在那些点上插入、修改或替换行为，而无需改变主流程代码。就像在流水线上某个环节设置了一个“检查点”或“插入器”。在 LangChain 的 Agent 执行循环中，比如 “模型调用前”“模型调用后”“工具调用前后” 都是可能挂钩子的点。

&emsp;&emsp;从模型调用角度看，Middleware 中常用的模型相关钩子包括 `before_model`、`modify_model_request`、`after_model`，也可以使用 `wrap_model_call` 包裹整次模型调用。它们对应“模型调用前 → 请求修改 → 模型返回后 / 整体包裹”的几个阶段。典型用途如下：

- `before_model`：总结或裁剪历史、注入系统指令、敏感信息脱敏、状态校验、条件分支跳转。
- `modify_model_request`：对即将发送的请求做精准改写，例如模型、参数、工具列表或 messages。
- `after_model`：做人审、输出校验、添加安全标签或生成可观测数据。
- `wrap_model_call`：在一次模型调用外层加壳，可动态换模型、改温度、做 A/B、重试、降级或路由。
- `wrap_tool_call`：统一处理工具调用的超时、重试、白名单、错误上报，或在人审批准前阻断高风险工具。

&emsp;&emsp;也就是说，中间件解决的是 Agent 工程化控制问题：主流程仍然是 ReAct，但每个关键节点都可以被观察、修改或拦截。


### 2. 模型动态选择

In [ ]:
from dotenv import find_dotenv, load_dotenv
import os
import getpass

# 1. 从 .env 文件加载环境变量
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

# 2. 获取 TAVILY_API_KEY
api_key = os.getenv("TAVILY_API_KEY")

# 3. 如果没获取到，则手动输入
if not api_key:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [ ]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=2, topic="general")
tools = [web_search]


本节统一使用 `create_model()` 创建模型，通过不同参数构造两个模型配置，用于演示中间件中的动态模型选择。默认模型是 `deepseek-v4-flash`，也可以手动改成 `deepseek-v4-pro`、`Qwen/Qwen3-8B` 或 `qwen/qwen3-coder:free`：

* **复杂问题 → 稳健配置**：较低`temperature`、更高`max_tokens`，适合多步骤、长上下文、证明与规划类问题。
* **简单问题 → 常规配置**：默认`temperature`和较低`max_tokens`，适合常规问答与闲聊。

In [ ]:
print("【要点】用 wrap_model_call 做动态模型路由：根据任务复杂度切换模型配置。")
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_core.messages import HumanMessage

# ① 准备同一模型的两种参数配置
basic_model = create_model("deepseek-v4-flash", temperature=0.3, max_tokens=1500)    # 简单问题：flash
careful_model = create_model("deepseek-v4-pro", temperature=0.1, max_tokens=3000)  # 复杂问题：pro


def _get_last_user_text(messages) -> str:
    """从消息列表中取最近一条用户消息文本（无则返回空串）"""
    for m in reversed(messages):
        if isinstance(m, HumanMessage):
            # content 可能是纯字符串或富内容；这里只处理为字符串的常见情况
            return m.content if isinstance(m.content, str) else ""
    return ""


@wrap_model_call
def dynamic_model_routing(request: ModelRequest, handler) -> ModelResponse:
    """
    根据对话复杂度动态选择同一模型的不同参数配置：
    - 复杂：低温度 + 更长输出
    - 简单：默认温度 + 较短输出
    """
    messages = request.state.get("messages", [])
    msg_count = len(messages)
    last_user = _get_last_user_text(messages)
    last_len = len(last_user)

    # 一些“复杂任务”关键词（可按需扩充）
    hard_keywords = ("证明", "推导", "严谨", "规划", "多步骤", "chain of thought",
                     "step-by-step", "reason step by step", "数学", "逻辑证明", "约束求解")

    # 简单的复杂度启发式：
    # 1) 历史消息较长   2) 最近用户输入很长   3) 出现复杂任务关键词
    is_hard = (
        msg_count > 10 or
        last_len > 120 or
        any(kw.lower() in last_user.lower() for kw in hard_keywords)
    )

    # 选择模型配置
    request.model = careful_model if is_hard else basic_model
    print(request.model)

    # 调用被包裹的下游（真正的模型调用）
    return handler(request)


# ② 创建 Agent（默认用常规配置，但运行时会被中间件按需替换）
agent = create_agent(
    model=basic_model,
    tools=tools,
    middleware=[dynamic_model_routing],
)

代码解释如下：

1. **同一模型的两种参数配置**
   本节统一使用 `create_model()` 创建同一模型的两种参数配置，分别实例化为：

* 常规配置：适合日常对话、直接问答、信息型问题。
* 稳健配置：适合多步骤推理、长上下文、复杂规划等任务。

2. **`@wrap_model_call` 中间件做“整体包裹”**

* 这个装饰器会把**一次模型调用**整体包起来，你能在这里**动态替换 `request.model`**。
* 调用 `handler(request)` 之前把 `request.model` 改成你想用的模型配置即可，实现**运行时路由**。
* 这种“外壳式”包装适合做**重试、超时、降级、动态路由**等控制。

3. **复杂度启发式（heuristics）**
   我们用了 3 条简单、可解释的启发式规则判断是否复杂：

* **会话长度**：`msg_count > 10` 说明上下文较长，信息综合度更高；
* **最近输入长度**：`last_len > 120` 说明问题描述较长，往往需要分解与规划；
* **关键词命中**：若用户写“证明/推导/规划/多步骤/step-by-step/chain of thought/数学/逻辑证明/约束求解”等词，倾向判定为复杂。

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
messages = {"messages": [{"role": "user", "content": "你好，请介绍下你自己。"}]}

In [ ]:
print("【要点】简单问题会走常规配置，观察中间件打印的模型选择。")
result = agent.invoke(messages)

In [ ]:
result

触发稳健配置调用：


In [ ]:
print("【要点】复杂问题会触发稳健配置，观察中间件如何替换 request.model。")
complex_question = """请帮我详细推理以下数学问题：
假设一个粒子以恒定加速度a沿直线运动，初速度为v0，位移为s。
推导出其速度与时间t的函数关系，并逐步解释每一步的物理意义。
"""

result = agent.invoke({
    "messages": [HumanMessage(content=complex_question)]
})

In [ ]:
result

### 3. 消息压缩

- 修剪消息（Trimming）

修剪（Trimming）是最轻量、最直接的压缩方式。
它的思想是：**只保留最近 N 条消息或 M 个 token 以内的上下文**，其余的自动裁剪掉。
这种方法通常配合 `@before_model` 钩子使用，在每次模型调用前计算历史消息数量或 token 长度；当接近模型的最大上下文限制（如 4k、16k、128k）时，就删除较早的对话，只保留关键的系统提示和最新几轮对话。
它的优点是实现简单、执行快速、成本可控，非常适合聊天型 Agent 或 RAG 问答系统；缺点是容易丢失远期上下文记忆。

In [ ]:
print("【要点】before_model 中间件用于模型调用前裁剪历史消息。")
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from typing import Any

# 初始化模型
model = create_model()

@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """在模型调用前修剪消息历史，只保留前1+后3条。"""
    messages = state["messages"]

    # 只有超过 4 条消息才裁剪
    if len(messages) <= 4:
        return None

    # 保留首条 System 提示 + 最近3条
    first_msg = messages[0]
    new_messages = [first_msg] + messages[-3:]

    print(f"✂️ 修剪消息：从 {len(messages)} 条 → {len(new_messages)} 条")

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

agent = create_agent(
    model=model,
    tools=tools,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "2"  
    }
}

In [ ]:
agent.invoke(
    {"messages": "你好，我叫陈明"},
    config
)

In [ ]:
agent.invoke(
    {"messages": "帮我写一句每日格言"},
    config
)

In [ ]:
agent.invoke(
    {"messages": "请介绍下你自己。"},
    config
)

In [ ]:
agent.invoke(
    {"messages": "你还记得我叫什么吗？"},
    config
)

- 删除消息（Deleting）

删除（Deleting）是一种更主动的上下文管理方式。
它允许开发者通过 `RemoveMessage` 机制，**精确指定要删除哪些消息**，例如清除最早的 2 条消息、只删除 tool 调用类消息、或直接重置整个会话。
这种方法通常配合 `@after_model` 钩子使用，也就是说在模型回复后清理历史消息，保证下次调用时上下文简洁干净。
删除策略通常用于长周期运行的 Agent 或工作流系统中，既能防止上下文爆炸，又能在关键节点（如新任务开始）重置状态，实现“会话阶段化”管理。

In [ ]:
print("【要点】after_model 中间件用于模型回复后清理旧消息。")
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langchain.messages import RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig

# 初始化模型
model = create_model()

@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    """模型调用后，删除最早的两条消息"""
    messages = state["messages"]

    if len(messages) > 4:
        removed = [m.id for m in messages[:2]]
        print(f"🧹 删除前两条消息，ID: {removed}")
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}
    return None

agent = create_agent(
    model,
    tools=[],
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)

In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "session-1"}}

agent.invoke({"messages": "你好，我叫小明"}, config)
agent.invoke({"messages": "请记住我的名字"}, config)
agent.invoke({"messages": "写一首关于春天的诗"}, config)
final = agent.invoke({"messages": "你还记得我叫什么吗？"}, config)

print("\n🤖 最终回答：", final["messages"][-1].content)

- 汇总消息（Summarization）

汇总（Summarization）是最“智能”的压缩策略。
与简单的修剪不同，它不是直接删除历史，而是**调用一个轻量模型自动总结过去的对话内容**，把长历史浓缩成短摘要，再拼接到当前上下文中。
LangChain 1.0 提供了现成的 `SummarizationMiddleware` 中间件，只需指定：

* 触发阈值 (`max_tokens_before_summary`)：超出后自动触发摘要；
* 摘要模型 (`model`)：通常使用成本更低的小模型；
* 保留的原始消息数量 (`messages_to_keep`)：例如只留最近 20 条原始消息。

这种方式兼顾了上下文连续性与效率，适合多轮对话、长期交互、或多 Agent 系统中做**会话记忆压缩**。

In [ ]:
print("【要点】SummarizationMiddleware 用摘要压缩长对话，保留近期原文消息。")
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

# 主模型（执行任务）
main_model = create_model(temperature=0.3, max_tokens=3000)

# 摘要模型（用于生成会话总结）
summary_model = create_model(temperature=0.1, max_tokens=1000)

checkpointer = InMemorySaver()

agent = create_agent(
    model=main_model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=summary_model,
            max_tokens_before_summary=3000,  # 超过3000token触发摘要
            messages_to_keep=10,             # 摘要后保留最近10条原始消息
        )
    ],
    checkpointer=checkpointer,
)

In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "summary-demo"}}

agent.invoke({"messages": "你好，我叫小明"}, config)
agent.invoke({"messages": "请写一首关于夏天的诗"}, config)
agent.invoke({"messages": "现在写一篇关于秋天的随笔"}, config)
agent.invoke({"messages": "再帮我写一段冬天的散文"}, config)
final = agent.invoke({"messages": "你能总结一下四季的特点吗？"}, config)

print("\n🤖 最终回答：", final["messages"][-1].content)

### 4. 借助中间件实现人在闭环（Human in the loop）

&emsp;&emsp;在自动化系统中，尤其是数据库写入、财务操作、邮件发送、文件修改等高风险任务里，完全让模型自主执行并不安全。LangChain 1.0 引入的 Human-in-the-Loop Middleware（人在闭环中间件）允许在关键节点暂停 Agent 执行，等待人工对模型准备执行的动作进行批准（approve）、编辑（edit）或拒绝（reject）。


In [ ]:
print("【要点】HumanInTheLoopMiddleware 会在高风险工具执行前暂停，等待人工批准。")
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# 创建 Agent，接入 HumanInTheLoopMiddleware
agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 拦截 Tavily 搜索工具执行前，要求人工确认
                "tavily_search": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "description": lambda tool_call, state, runtime: (
                        f"🔍 模型准备执行 Tavily 搜索：'{tool_call.get('args', {}).get('query', '')}'"
                    ),
                }
            },
            description_prefix="⚠️ 工具执行需要人工审批"
        )
    ],
)


In [ ]:
config = {
    "configurable": {
        "thread_id": "23"  
    }
}

In [ ]:
result = agent.invoke(
    {"messages": "你好，请问2024年诺贝尔奖得主有哪些？"},
    config
)

In [ ]:
result

人在闭环功能在部署后可以直观看到效果：

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028194735535.png" alt="image-20251028194735535" style="zoom:50%;" />

<center><img src="https://ml2022.oss-cn-hangzhou.aliyuncs.com/img/image-20251028195109084.png" alt="image-20251028195109084" style="zoom:50%;" />

# 第六章 综合实战：新闻 + 天气 + 文件 Agent

&emsp;&emsp;前面几章分别学习了模型调用、工具接入、MCP、ReAct、记忆和中间件。最后这一章把这些元素合在一起，构建一个更接近真实项目的综合 Agent。

这个 Agent 具备三类能力：

| 能力 | 工具来源 | 用途 |
| --- | --- | --- |
| 查新闻 | `TavilySearch` | 查询实时新闻、热点和公开网页信息 |
| 查天气 | `OpenWeatherMCPServer` | 通过 MCP 服务查询城市实时天气 |
| 操作文件 | `SecureFileMCPServer` | 读取 `files/` 目录中的文件，理解内容，重命名为合适的文件名和后缀，并支持回退 |

&emsp;&emsp;本章的文件实战会使用当前目录下的 `files/` 样例文件夹。这个文件夹中有 `a.txt`、`b.txt`、`c.txt`、`d.txt`、`e.txt` 五个文件，它们的内容分别是不同编程语言的文件读写示例。Agent 的任务是读取内容、判断语言、生成合适文件名，然后通过 MCP 执行重命名。


## 0. 本章结构：分步执行 → 一把执行

&emsp;&emsp;本章把前面五章的模型、工具、MCP、ReAct、记忆、中间件串成一个完整 Agent。为了让每个环节都能单独观察和理解，本章采用**先分步、后合并**的组织方式：先完成读取和规划，再集中演示文件重命名、结果查看与回退，最后给出一把执行版。

1. **实战目标与依赖条件**：确认 `.env` 中已配置所需 API Key。
2. **启动天气 MCP 与文件 MCP**：把两个 MCP 服务在本地启动起来。
3. **加载新闻、天气和文件工具**：把 `TavilySearch`、天气 MCP、文件 MCP 加载为 LangChain Tool。
4. **先直接验证工具链路**：直接调用关键工具，确认新闻、天气、文件链路正常。
5. **文件实战：读取 files/ 下全部文件**：列出并读取样例文件，理解内容，形成重命名计划。
6. **创建综合 Agent**：接入记忆、`HumanInTheLoopMiddleware` 审批和文件安全系统提示。
7. **任务一：查新闻 + 查天气 + 读文件 + 生成重命名计划**：只做读取和规划，不修改文件。
8. **任务二：执行文件重命名**：在用户明确授权后调用 `file_rename_file`，提供自动审批版和 `input()` 人工审批版两种演示方式。
9. **查看重命名结果和历史记录**：用 `file_list_files` 和 `file_view_rename_history` 确认文件已经改名，并查看可回退的 rename history。
10. **验证记忆**：复用同一 `thread_id`，确认 Agent 能延续前文上下文。
11. **任务三：回退文件重命名**：基于 rename history 生成回退计划，明确打印要回退几个文件、每次从哪个文件回退到哪个文件，并提供自动审批版和 `input()` 人工审批版。
12. **完整代码：一把执行综合 Agent**：把上述步骤合并为一个可独立运行的 cell。
13. **关闭本章启动的 MCP 服务**。
14. **本章小结**。

> 说明：任务二和任务三中的 A/B 版本用于对比自动审批与人工审批，课堂演示时可以按需要选择其中一种路径运行。分步执行中使用的工具名、函数名、Agent 配置和 HITL 逻辑，与最后的**一把执行版**保持一致，便于对照理解。


## 1. 实战目标与依赖条件

运行本章前，需要确认 `.env` 中至少配置：

```bash
DEEPSEEK_API_KEY=你的 DeepSeek Key
SILICONFLOW_API_KEY=你的 SiliconFlow Key
OPENROUTER_API_KEY=你的 OpenRouter Key
TAVILY_API_KEY=你的 Tavily Key
OPENWEATHER_API_KEY=你的 OpenWeather Key
```

其中 `DEEPSEEK_API_KEY` 用于 `deepseek-v4-flash` 和 `deepseek-v4-pro`，`SILICONFLOW_API_KEY` 用于 `Qwen/Qwen3-8B`，`OPENROUTER_API_KEY` 用于 `qwen/qwen3-coder:free`，`TAVILY_API_KEY` 用于查新闻，`OPENWEATHER_API_KEY` 用于天气 MCP 查询。如果暂时没有 OpenWeather Key，也可以先运行工具加载、文件 MCP 和文件重命名单元。


In [ ]:
print("【要点】综合实战准备：加载环境变量、定义 create_model()、定位 MCP 脚本和 files 样例目录。")
import getpass
import httpx
import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

# 清理系统代理环境变量，避免本地代理配置影响 API 请求
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy",
]:
    os.environ.pop(k, None)

# 加载环境变量；优先尝试当前目录 .env，否则尝试项目根目录 .env
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

# 本章默认使用 vLLM Qwen 122B；切换 DeepSeek 时再配置 DEEPSEEK_API_KEY。
for env_name in ["vLLM_QWEN122B_API_KEY", "TAVILY_API_KEY", "OPENWEATHER_API_KEY"]:
    if not os.environ.get(env_name):
        os.environ[env_name] = getpass.getpass(f"{env_name}:\n")

def resolve_course_dir() -> Path:
    """兼容从项目根目录或 notebook 目录启动。"""
    cwd = Path.cwd().resolve()
    relative_course_dir = Path("04_综合实战/06_综合实战-Langchain")
    candidates = [cwd / relative_course_dir, cwd]
    for candidate in candidates:
        if (candidate / "mcp").is_dir() and (candidate / "files").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        f"无法定位课程目录；cwd={cwd}，请从项目根目录或该 notebook 所在目录运行。"
    )


COURSE_DIR = resolve_course_dir()
WEATHER_MCP_SCRIPT = COURSE_DIR / "mcp/01_weather-mcp-server" / "openweather_mcp_server.py"
FILE_MCP_SCRIPT = COURSE_DIR / "mcp/02_file-mcp-server" / "secure_file_mcp_server.py"
DEFAULT_FILES_OPERATION_DIR = (COURSE_DIR / "files").resolve()
operation_dir_input = os.getenv("LANGCHAIN_FILES_OPERATION_DIR", "").strip()
if not operation_dir_input:
    operation_dir_input = input(
        f"请输入要操作的文件夹绝对路径 [{DEFAULT_FILES_OPERATION_DIR}]: "
    ).strip() or str(DEFAULT_FILES_OPERATION_DIR)

FILES_OPERATION_DIR = Path(operation_dir_input).expanduser()
if not FILES_OPERATION_DIR.is_absolute():
    raise ValueError(f"请输入绝对路径: {FILES_OPERATION_DIR}")
FILES_OPERATION_DIR = FILES_OPERATION_DIR.resolve()
if not FILES_OPERATION_DIR.is_dir():
    raise NotADirectoryError(f"文件操作目录不存在或不是目录: {FILES_OPERATION_DIR}")
FILE_TOOL_DIRECTORY = str(FILES_OPERATION_DIR)

print(f"[配置] 课程目录: {COURSE_DIR}")
print(f"[配置] 文件操作目录: {FILES_OPERATION_DIR}")
print(f"[配置] 目录存在: {FILES_OPERATION_DIR.is_dir()}")

INTEGRATED_MCP_PROCESSES = {}


MODEL_CONFIGS = {
    "deepseek-v4-flash": {
        "model": "deepseek-v4-flash",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的轻量快速模型",
    },
    "deepseek-v4-pro": {
        "model": "deepseek-v4-pro",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的更强能力模型",
    },
    "vllm-qwen122b": {
        "model": os.getenv("vLLM_QWEN122B_MODEL_ID"),
        "base_url": os.getenv("vLLM_QWEN122B_BASE_URL"),
        "api_key_env": "vLLM_QWEN122B_API_KEY",
        "description": "vLLM 部署的 Qwen 122B 模型（默认）",
    },
    "Qwen/Qwen3-8B": {
        "model": "Qwen/Qwen3-8B",
        "base_url": "https://api.siliconflow.cn/v1",
        "api_key_env": "SILICONFLOW_API_KEY",
        "description": "SiliconFlow 上的 Qwen/Qwen3-8B 模型",
    },
    "qwen/qwen3-coder:free": {
        "model": "qwen/qwen3-coder:free",
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
        "description": "OpenRouter 上的 Qwen3 Coder 免费模型",
    },
}

DEFAULT_MODEL_NAME = "vllm-qwen122b"
# 切换到 DeepSeek：create_model("deepseek-v4-flash") 或 create_model("deepseek-v4-pro")


def create_model(
    model_name: str = DEFAULT_MODEL_NAME,
    temperature: float = 0.3,
    max_tokens: int = 10000,
    verbose: bool = True,
):
    """创建课程统一使用的 OpenAI 兼容聊天模型。

    支持五种选择：
    - vllm-qwen122b：从 .env 读取 vLLM Qwen 122B 配置（默认）
    - deepseek-v4-flash：DeepSeek 兼容 OpenAI API 模型服务
    - deepseek-v4-pro：DeepSeek 兼容 OpenAI API 模型服务
    - Qwen/Qwen3-8B：SiliconFlow 上的 Qwen 模型
    - qwen/qwen3-coder:free：OpenRouter 上的 Qwen3 Coder 免费模型
    """
    if model_name not in MODEL_CONFIGS:
        available = ", ".join(MODEL_CONFIGS)
        raise ValueError(f"不支持的 model_name: {model_name}。可选值: {available}")

    config = MODEL_CONFIGS[model_name]
    if not config.get("model") or not config.get("base_url"):
        raise ValueError(f"模型 {model_name} 缺少 model 或 base_url 配置，请检查 .env。")
    api_key = os.getenv(config["api_key_env"])
    if not api_key:
        raise ValueError(f"缺少环境变量 {config['api_key_env']}，请先在 .env 中配置。")

    if verbose:
        print(f"创建模型: {model_name} | {config['description']}")

    return init_chat_model(
        model=config["model"],
        model_provider="openai",
        base_url=config["base_url"],
        api_key=api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=temperature,
        max_tokens=max_tokens,
    )


## 2. 启动天气 MCP 与文件 MCP

&emsp;&emsp;综合 Agent 会把天气和文件能力都作为 MCP 工具接入。这里继续使用 `streamable-http` 方式启动本地 MCP Server：

- 天气 MCP：`http://127.0.0.1:16681/mcp`
- 文件 MCP：`http://127.0.0.1:16682/mcp`

文件 MCP 是通用文件服务，不设允许根目录。运行时由用户输入本次综合实战要操作的**绝对目录**；这只是 Agent 任务的目标目录，不是 MCP 的访问边界。按回车可使用默认的 `04_综合实战/06_综合实战-Langchain/files`；无人值守运行时可通过 `LANGCHAIN_FILES_OPERATION_DIR` 传入。


In [ ]:
print("【要点】启动综合实战所需的天气 MCP 和文件 MCP。")
def is_port_open(port: int, host: str = "127.0.0.1", timeout: float = 0.5) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def start_mcp_server(
    name: str, script_path: Path, port: int, extra_env: dict | None = None, reuse_existing: bool = True
):
    """启动本地 MCP Server；可选是否复用已占用端口的服务。"""
    if is_port_open(port):
        if reuse_existing:
            return {"name": name, "port": port, "status": "already_running"}
        raise RuntimeError(
            f"{name} MCP 端口 {port} 已被占用，无法确认正在运行的是本章服务。"
            "请先运行本章的『关闭 MCP 服务』单元，或停止占用该端口的旧服务。"
        )

    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)

    process = subprocess.Popen(
        [sys.executable, str(script_path)],
        cwd=str(script_path.parent),
        env=env,
        # 不使用未持续消费的 PIPE，避免 MCP 日志写满缓冲区后阻塞。
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        text=True,
    )

    for _ in range(40):
        if is_port_open(port):
            INTEGRATED_MCP_PROCESSES[name] = process
            return {"name": name, "port": port, "status": "started", "pid": process.pid}
        if process.poll() is not None:
            stderr = process.stderr.read() if process.stderr else ""
            raise RuntimeError(f"{name} 启动失败：{stderr}")
        time.sleep(0.25)

    process.terminate()
    raise TimeoutError(f"{name} 在端口 {port} 上启动超时。")


integrated_weather_status = start_mcp_server(
    name="weather",
    script_path=WEATHER_MCP_SCRIPT,
    port=16681,
    extra_env={"WEATHER_MCP_PORT": "16681"},
)

integrated_file_status = start_mcp_server(
    name="file",
    script_path=FILE_MCP_SCRIPT,
    port=16682,
    reuse_existing=False,
    extra_env={
        "FILE_MCP_MAX_READ_BYTES": "200000",
    },
)

print("[日志] 文件 MCP 不设允许根目录")
print(f"[日志] 本章指定操作目录: {FILES_OPERATION_DIR}")
integrated_weather_status, integrated_file_status


## 3. 加载新闻、天气和文件工具

&emsp;&emsp;这里把不同来源的工具放在同一个 `tools` 列表里：`TavilySearch` 负责查新闻，天气和文件能力来自 MCP Server。只要最终转换成 LangChain Tool，`create_agent` 的接入方式就是一致的。

&emsp;&emsp;两个 MCP 服务都有 `ping` 这类同名工具，因此 `MultiServerMCPClient` 使用 `tool_name_prefix=True`，让工具名变成 `weather_get_weather`、`file_read_file`、`file_rename_file` 这样的稳定形式。


In [ ]:
print("【要点】加载新闻搜索、天气 MCP、文件 MCP 三类工具，并合并成 integrated_tools。")
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_tavily import TavilySearch

news_search = TavilySearch(
    max_results=3,
    topic="news",
)

integrated_mcp_client = MultiServerMCPClient(
    {
        "weather": {
            "url": "http://127.0.0.1:16681/mcp",
            "transport": "http",
        },
        "file": {
            "url": "http://127.0.0.1:16682/mcp",
            "transport": "http",
        },
    },
    tool_name_prefix=True,
)

integrated_mcp_tools = await integrated_mcp_client.get_tools()
integrated_tools = [news_search, *integrated_mcp_tools]
integrated_tool_by_name = {tool.name: tool for tool in integrated_tools}

[(tool.name, tool.description[:80]) for tool in integrated_tools]


## 4. 先直接验证工具链路

&emsp;&emsp;综合 Agent 一旦失败，可能是模型、工具、API Key、MCP 服务或参数格式中的任何一环有问题。因此在教学和开发中，建议先直接调用关键工具，确认工具本身可以独立运行。


In [ ]:
print("【要点】先直接验证关键工具链路：天气配置、文件根目录、files 列表。")
def parse_mcp_tool_result(result):
    """把 langchain-mcp-adapters 返回的 content blocks 尽量解析为 Python 对象。"""
    if isinstance(result, list) and result and isinstance(result[0], dict):
        text = result[0].get("text", "")
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {"raw_text": text}
    return result


weather_config = parse_mcp_tool_result(
    await integrated_tool_by_name["weather_check_config"].ainvoke({})
)
file_roots = parse_mcp_tool_result(
    await integrated_tool_by_name["file_list_allowed_roots"].ainvoke({})
)
files_listing = parse_mcp_tool_result(
    await integrated_tool_by_name["file_list_files"].ainvoke({"directory": FILE_TOOL_DIRECTORY})
)

print(f"[验证] 期望操作目录: {FILES_OPERATION_DIR}")
print(f"[验证] MCP 允许根目录: {file_roots}")
weather_config, file_roots, files_listing


In [ ]:
# 直接验证新闻搜索工具。这个单元会访问 Tavily API。
news_search.invoke({"query": "人工智能 最新新闻"})


In [ ]:
# 直接验证天气 MCP。这个单元会访问 OpenWeather API。
parse_mcp_tool_result(
    await integrated_tool_by_name["weather_get_weather"].ainvoke({"location": "Beijing"})
)


## 5. 文件实战：读取 files/ 下全部文件

&emsp;&emsp;文件实战先从最小动作开始：列出 `files/` 目录，再用 `file_read_file` 逐个读取内容。这样把"读取"和"理解/重命名"拆开，便于教学时单独调试文件 MCP 链路。

&emsp;&emsp;`files/` 样例目录中有 `a.txt` 到 `e.txt` 五个文件，内容分别是 Python、Go、Java、C 和 JavaScript 的文件读写示例。Agent 后续会根据这些内容判断语言并生成合适的新文件名。


In [ ]:
print("【要点】读取 files/ 下每个文件的内容预览，为后续重命名计划提供依据。")
files_listing = parse_mcp_tool_result(
    await integrated_tool_by_name["file_list_files"].ainvoke({"directory": FILE_TOOL_DIRECTORY})
)

file_contents = []
for item in files_listing.get("items", []):
    if not item.get("is_file"):
        continue
    name = item["name"]
    content_result = parse_mcp_tool_result(
        await integrated_tool_by_name["file_read_file"].ainvoke({"filepath": f"{FILE_TOOL_DIRECTORY}/{name}"})
    )
    file_contents.append(
        {
            "name": name,
            "size_bytes": item.get("size_bytes"),
            "content_preview": content_result.get("content", "")[:500],
        }
    )

file_contents


### 5.1 样例文件期望识别方向

&emsp;&emsp;下面是这批样例文件的期望识别方向。真实项目中可以让 Agent 自己生成这张计划表；教学时先给出参考答案，学习者更容易理解"为什么要这样改名"。


In [ ]:
expected_rename_plan = [
    {"old_path": str(FILES_OPERATION_DIR / "a.txt"), "new_name": "python_file_io_demo.py", "reason": "内容使用 Python open/read/write/append。"},
    {"old_path": str(FILES_OPERATION_DIR / "b.txt"), "new_name": "go_file_io_demo.go", "reason": "内容包含 package main、import 和 os.OpenFile。"},
    {"old_path": str(FILES_OPERATION_DIR / "c.txt"), "new_name": "java_file_io_demo.java", "reason": "内容包含 Java class Main、FileWriter 和 IOException。"},
    {"old_path": str(FILES_OPERATION_DIR / "d.txt"), "new_name": "c_file_io_demo.c", "reason": "内容包含 #include <stdio.h>、fopen、fprintf。"},
    {"old_path": str(FILES_OPERATION_DIR / "e.txt"), "new_name": "javascript_file_io_demo.js", "reason": "内容使用 Node.js fs.writeFile 和 fs.readFile。"},
]

expected_rename_plan


## 6. 创建带记忆、文件安全约束与 HITL 审批的综合 Agent

&emsp;&emsp;这一版 Agent 同时接入三类工具，并使用 `InMemorySaver` 保存同一个 `thread_id` 下的对话状态。和前面的版本相比，这里新增了 `HumanInTheLoopMiddleware`：

1. `file_list_files`、`file_read_file`、`file_get_file_info` 等只读工具可以自动执行。
2. `file_rename_file` 会修改文件名，执行前必须人工审批。
3. `file_restore_last_rename` 会回退文件名，也属于文件修改操作，执行前必须人工审批。
4. 审批人可以选择 `approve`、`edit` 或 `reject`；其中回退工具没有参数可编辑，所以只开放 `approve` 和 `reject`。

&emsp;&emsp;系统提示仍然负责告诉模型“什么能做、什么不能做”，HITL 中间件负责在工具真正执行前拦截高风险动作。二者职责不同：提示词是行为约束，中间件是运行时防线。


In [ ]:
print("【要点】创建综合 Agent：接入记忆、文件安全约束和 HITL 审批。")
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

integrated_model = create_model(
    temperature=0.1,
    max_tokens=10000,
)

integrated_checkpointer = InMemorySaver()


def describe_file_modification_request(tool_call, state, runtime) -> str:
    """为文件修改类工具生成更适合人工审批的说明。"""
    tool_name = tool_call["name"]
    tool_args = tool_call.get("args", {})

    if tool_name == "file_rename_file":
        return (
            "准备执行文件重命名，请人工确认：\n"
            f"- old_path: {tool_args.get('old_path')}\n"
            f"- new_name: {tool_args.get('new_name')}\n"
            f"- base_dir: {tool_args.get('base_dir') or '(默认允许根目录)'}\n"
            "请确认该操作只作用于 files/ 样例目录，且新文件名和后缀符合教学目标。"
        )

    if tool_name == "file_restore_last_rename":
        return (
            "准备回退最近一次文件重命名，请人工确认：\n"
            f"- filter_dir: {tool_args.get('filter_dir') or '(all history)'}\n"
            "- 该操作会把最近一次 rename 历史恢复到旧文件名。\n"
            "- 如果需要连续回退多个文件，Agent 可能会多次请求审批。"
        )

    return f"准备执行文件修改工具 {tool_name}，参数：{tool_args}"


def get_hitl_action_requests(agent_result: dict) -> list[dict]:
    """从 Agent 中断结果中提取待人工审批的工具动作。"""
    interrupts = agent_result.get("__interrupt__", ())
    if not interrupts:
        return []
    hitl_request = interrupts[0].value
    return hitl_request.get("action_requests", [])


integrated_agent = create_agent(
    model=integrated_model,
    tools=integrated_tools,
    checkpointer=integrated_checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "file_rename_file": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "description": describe_file_modification_request,
                },
                "file_restore_last_rename": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": describe_file_modification_request,
                },
            },
            description_prefix="文件修改操作需要人工审批",
        )
    ],
    system_prompt=(
        "你是一个面向课程实战的综合 Agent，可以查新闻、查天气、操作课程 files 目录中的样例文件。"
        "当用户询问最新新闻、近期动态或需要外部事实核实时，优先调用 tavily_search。"
        "当用户询问天气时，优先调用 weather_get_weather，参数名使用 location，城市名使用英文，例如 Beijing、Shanghai。"
        "当用户要求查看文件时，使用 file_list_files、file_read_file、file_get_file_info 等文件 MCP 工具。"
        f"文件 MCP 是通用服务；用户本次指定的操作目录是 {FILES_OPERATION_DIR}。"
        f"调用文件工具时，directory 和 filter_dir 使用 {FILE_TOOL_DIRECTORY}，filepath/old_path 使用该目录下的绝对路径。"
        f"调用 file_restore_last_rename 时必须传入 filter_dir={FILE_TOOL_DIRECTORY}，只回退当前选定目录的历史。"
        "只有用户明确授权整理、重命名或回退文件时，才允许调用 file_rename_file 或 file_restore_last_rename。"
        "重命名时必须先读取文件内容，根据内容判断语言和用途，再生成英文小写、下划线分隔、后缀正确的新文件名。"
        "执行文件重命名或回退前，运行时会要求人工审批；如果审批被拒绝，不要重复调用同一个修改工具。"
        "回答时用中文说明调用了哪些工具、识别依据、重命名结果，以及如何回退。"
    ),
)

integrated_agent_config = {
    "configurable": {
        "thread_id": "integrated-news-weather-file-agent-hitl"
    }
}


## 7. 任务一：查新闻、查天气、读文件、生成重命名计划

&emsp;&emsp;这个任务会促使 Agent 使用多个工具完成一个小型工作流：查新闻、查天气、读取 `files/` 目录、理解每个文件内容，并**生成重命名计划**。

&emsp;&emsp;注意：本单元只要求 Agent 生成计划并总结，不调用 `file_rename_file`。这样可以让学习者先观察 Agent 的推理和规划能力，再把文件修改动作放到下一个任务中单独演练。


In [ ]:
print("【要点】任务一：查新闻、查天气、读文件并生成重命名计划（不执行修改）。")

task1_result = await integrated_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "请完成一个综合任务："
                    "1）查询一条最新的人工智能相关新闻；"
                    "2）查询 Beijing 当前天气；"
                    "3）列出 files/ 目录，逐个读取里面的文件；"
                    "4）根据每个文件内容判断编程语言和用途；"
                    "5）给出重命名计划：为每个文件生成合适的英文文件名，并使用正确的代码文件后缀，例如 .py、.go、.java、.c、.js；"
                    "6）用中文总结新闻、天气、文件识别依据和拟定的重命名计划。"
                    "注意：现在只需要生成计划，不要调用 file_rename_file 实际修改文件。"
                ),
            }
        ]
    },
    config=integrated_agent_config,
)

# 如果模型意外触发了文件修改工具的 HITL 中断，给出提示而不是直接打印最后一条消息。
if "__interrupt__" in task1_result:
    pending = get_hitl_action_requests(task1_result)
    print(f"⚠️ 任务一意外触发了 {len(pending)} 个待审批动作，请在继续前先处理这些审批：")
    for action in pending:
        print(f"  - {action.get('name')}: {action.get('args')}")
else:
    print(task1_result["messages"][-1].content)


## 8. 任务二：执行文件重命名

&emsp;&emsp;在任务一中 Agent 已经读取了文件并生成了重命名计划。本单元在用户的明确授权下，让 Agent 调用 `file_rename_file` 实际执行重命名。

&emsp;&emsp;由于第 6 节已经接入 `HumanInTheLoopMiddleware`，当 Agent 准备调用 `file_rename_file` 时，运行时会暂停并返回 `__interrupt__`。本单元把 `AUTO_APPROVE_FILE_CHANGES` 设置为 `True`，**自动批准**所有待审批动作，适合无人值守的课堂演示或快速验证链路。

> 提示：如果希望体验逐条人工确认，请跳到后面的「任务二 B：执行文件重命名（input() 人工审批版）」。


### A: 自动审批版

In [ ]:
# print("【要点】任务二 A：自动审批执行文件重命名。")

# AUTO_APPROVE_FILE_CHANGES = True

# task2_auto_result = await integrated_agent.ainvoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": (
#                     "我已经确认了你的重命名计划。现在我明确授权你执行文件重命名："
#                     "请根据刚才对 files/ 目录中文件内容的判断，调用 file_rename_file 把每个 .txt 文件重命名为合适的英文代码文件名。"
#                     "每次重命名前都需要人工审批，如果审批被拒绝，请停止并说明原因。"
#                 ),
#             }
#         ]
#     },
#     config=integrated_agent_config,
# )

# approved_rename_actions_auto = []
# while "__interrupt__" in task2_auto_result:
#     current_actions = get_hitl_action_requests(task2_auto_result)
#     approved_rename_actions_auto.extend(current_actions)
#     approval_decisions = [{"type": "approve"} for _ in current_actions]
#     print(f"自动批准 {len(current_actions)} 个重命名动作。")
#     task2_auto_result = await integrated_agent.ainvoke(
#         Command(resume={"decisions": approval_decisions}),
#         config=integrated_agent_config,
#     )

# print(f"已自动审批通过 {len(approved_rename_actions_auto)} 个重命名动作。")
# print(task2_auto_result["messages"][-1].content)


### B：input() 人工审批版

&emsp;&emsp;这一小节与「任务二 A」做对比：同样的重命名操作，但把审批方式改为**命令行 input() 逐条人工确认**。运行时会依次打印每个待审批的 `file_rename_file` 动作，输入 `a` 批准、`r` 拒绝（默认拒绝）。

&emsp;&emsp;如果刚才已经跑过任务二 A 和任务三 A，文件应该已经恢复到原始 `a.txt` ~ `e.txt` 状态，可以直接运行本小节；如果跳过 A 直接跑 B，Agent 也会根据当前文件状态处理。


In [ ]:
print("【要点】任务二 B：input() 人工审批执行文件重命名。")


def review_actions_terminal(actions: list[dict]) -> list[dict]:
    """命令行 HITL 审批：逐条确认每个文件修改动作。"""
    decisions = []
    for index, action in enumerate(actions, start=1):
        name = action.get("name", "unknown_tool")
        args = action.get("args", {})
        desc = action.get("description", "")
        print(f"\n审批 {index}/{len(actions)}: {name}")
        print(f"  参数: {args}")
        print(f"  说明: {desc[:200]}")
        choice = input("请输入 a=批准, r=拒绝 [r]: ").strip().lower()
        if choice == "a":
            decisions.append({"type": "approve"})
        else:
            decisions.append({"type": "reject", "message": f"人工审批拒绝执行 {name}。"})
    return decisions


task2_manual_result = await integrated_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "我已经确认了你的重命名计划。现在我明确授权你执行文件重命名："
                    "请根据刚才对 files/ 目录中文件内容的判断，调用 file_rename_file 把每个 .txt 文件重命名为合适的英文代码文件名。"
                    "每次重命名前都需要人工审批，如果审批被拒绝，请停止并说明原因。"
                ),
            }
        ]
    },
    config=integrated_agent_config,
)

approved_rename_actions_manual = []
while "__interrupt__" in task2_manual_result:
    current_actions = get_hitl_action_requests(task2_manual_result)
    approved_rename_actions_manual.extend(current_actions)
    approval_decisions = review_actions_terminal(current_actions)
    task2_manual_result = await integrated_agent.ainvoke(
        Command(resume={"decisions": approval_decisions}),
        config=integrated_agent_config,
    )

print(f"已人工审批处理 {len(approved_rename_actions_manual)} 个重命名动作。")
print(task2_manual_result["messages"][-1].content)


## 9. 查看重命名结果和历史记录

&emsp;&emsp;自动审批和人工审批两个流程都跑完后，用 MCP 再次查看 `files/` 目录和重命名历史。历史记录由 `SecureFileMCPServer` 维护，可用于追踪哪些文件被修改过。

&emsp;&emsp;如果前面完整地执行了任务二 A/B 和任务三 A/B，此时 `files/` 应该已经恢复到原始的 `a.txt` ~ `e.txt`；如果只执行了部分流程，结果会对应保留当时的文件状态。


In [ ]:
print("【要点】查看文件重命名结果和 rename history，确认修改可追踪。")
renamed_files_listing = parse_mcp_tool_result(
    await integrated_tool_by_name["file_list_files"].ainvoke({"directory": FILE_TOOL_DIRECTORY})
)

rename_history = parse_mcp_tool_result(
    await integrated_tool_by_name["file_view_rename_history"].ainvoke({"filter_dir": FILE_TOOL_DIRECTORY})
)

renamed_files_listing, rename_history


## 10. 验证记忆：让 Agent 延续上一轮上下文

&emsp;&emsp;由于本章创建 Agent 时传入了 `checkpointer`，只要继续使用同一个 `thread_id`，Agent 就能访问上一轮对话中的上下文。下面的问题没有重新说明所有文件内容，但 Agent 应该能基于上一轮整理结果继续回答。


In [ ]:
print("【要点】验证记忆：复用同一 thread_id，让 Agent 延续上一轮整理上下文。")
memory_result = await integrated_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "基于刚才 files 目录的整理结果，请说明每个新文件名为什么比原来的 a.txt、b.txt 这类名字更适合教学。",
            }
        ]
    },
    config=integrated_agent_config,
)

memory_result["messages"][-1].content


## 11. 任务三：回退文件重命名

&emsp;&emsp;`SecureFileMCPServer` 会把每一次 `file_rename_file` 写入重命名历史，`file_restore_last_rename` 每次只回退最近一条历史。这个设计故意保持“单步可审计”：如果前面重命名了 5 个文件，就需要连续调用 5 次回退工具。

&emsp;&emsp;本节先演示自动审批版：Agent 负责判断还需要回退几次，运行时自动批准所有 `file_restore_last_rename` 审批请求，直到 `files/` 目录相关重命名历史被清空或 Agent 停止调用回退工具。


### A：自动审批版

In [ ]:
# print("【要点】任务三 A：自动审批回退文件重命名。")

# AUTO_APPROVE_FILE_CHANGES = True


# def build_restore_plan(history_result: dict) -> list[dict]:
#     """把 rename history 转成回退计划。最近的重命名最先回退。"""
#     items = history_result.get("items", []) if history_result.get("ok") else []
#     return list(reversed(items))


# def print_restore_plan(plan: list[dict], title: str = "回退计划") -> None:
#     print(f"{title}：共需要回退 {len(plan)} 个文件。")
#     if not plan:
#         print("当前没有 files/ 目录相关的重命名历史，无需回退。")
#         return
#     for index, item in enumerate(plan, start=1):
#         print(f"  {index}. {item.get('new_path')} -> {item.get('old_path')}")


# history_before_restore_auto = parse_mcp_tool_result(
#     await integrated_tool_by_name["file_view_rename_history"].ainvoke({"filter_dir": "files"})
# )
# restore_plan_auto = build_restore_plan(history_before_restore_auto)
# print_restore_plan(restore_plan_auto, title="任务三 A 回退计划")

# task3_auto_result = await integrated_agent.ainvoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": (
#                     "我明确授权你回退刚才对 files/ 目录执行的文件重命名操作。"
#                     "请先查看 file_view_rename_history，只处理 filter_dir 为 files 的历史；"
#                     "然后连续调用 file_restore_last_rename，每次恢复最近一条重命名记录，"
#                     "直到 files/ 目录相关重命名历史为空。"
#                     "每次回退前都需要人工审批，如果审批被拒绝，请停止并说明原因。"
#                 ),
#             }
#         ]
#     },
#     config=integrated_agent_config,
# )

# approved_restore_actions_auto = []
# restore_step_auto = 0
# while "__interrupt__" in task3_auto_result:
#     current_actions = get_hitl_action_requests(task3_auto_result)
#     if not current_actions:
#         break

#     approval_decisions = []
#     for action in current_actions:
#         if action.get("name") == "file_restore_last_rename":
#             restore_step_auto += 1
#             plan_item = restore_plan_auto[restore_step_auto - 1] if restore_step_auto <= len(restore_plan_auto) else {}
#             print(
#                 f"自动批准回退 {restore_step_auto}/{len(restore_plan_auto)}："
#                 f"{plan_item.get('new_path', '(未知当前文件)')} -> {plan_item.get('old_path', '(未知恢复目标)')}"
#             )
#         else:
#             print(f"自动批准工具动作：{action.get('name')}")
#         approved_restore_actions_auto.append(action)
#         approval_decisions.append({"type": "approve"})

#     task3_auto_result = await integrated_agent.ainvoke(
#         Command(resume={"decisions": approval_decisions}),
#         config=integrated_agent_config,
#     )

# print(f"已自动审批通过 {len(approved_restore_actions_auto)} 个回退动作。")
# print(task3_auto_result["messages"][-1].content)

# files_after_restore_auto = parse_mcp_tool_result(
#     await integrated_tool_by_name["file_list_files"].ainvoke({"directory": "files"})
# )
# history_after_restore_auto = parse_mcp_tool_result(
#     await integrated_tool_by_name["file_view_rename_history"].ainvoke({"filter_dir": "files"})
# )

# files_after_restore_auto, history_after_restore_auto


### B：input() 人工审批版

&emsp;&emsp;这一小节与「任务三 A」做对比：仍然通过 Agent 调用 `file_restore_last_rename`，但每次回退前都在命令行里逐条确认。输入 `a` 批准，输入 `r` 或直接回车拒绝。

&emsp;&emsp;如果刚运行过「任务二 B」，这里会把人工审批重命名产生的历史逐条恢复；如果当前没有重命名历史，Agent 应该说明无需回退。


In [ ]:
print("【要点】任务三 B：input() 人工审批回退文件重命名。")


if "build_restore_plan" not in globals():
    def build_restore_plan(history_result: dict) -> list[dict]:
        """把 rename historyi 转成回退计划。最近的重命名最先回退。"""
        items = history_result.get("items", []) if history_result.get("ok") else []
        return list(reversed(items))


if "print_restore_plan" not in globals():
    def print_restore_plan(plan: list[dict], title: str = "回退计划") -> None:
        print(f"{title}：共需要回退 {len(plan)} 个文件。")
        if not plan:
            print("当前没有 files/ 目录相关的重命名历史，无需回退。")
            return
        for index, item in enumerate(plan, start=1):
            print(f"  {index}. {item.get('new_path')} -> {item.get('old_path')}")

history_before_restore_manual = parse_mcp_tool_result(
    await integrated_tool_by_name["file_view_rename_history"].ainvoke({"filter_dir": FILE_TOOL_DIRECTORY})
)
restore_plan_manual = build_restore_plan(history_before_restore_manual)
print_restore_plan(restore_plan_manual, title="任务三 B 回退计划")

restore_results_manual = []
for restore_step_manual, plan_item in enumerate(restore_plan_manual, start=1):
    print(
        f"\n准备人工审批回退 {restore_step_manual}/{len(restore_plan_manual)}："
        f"{plan_item.get('new_path')} -> {plan_item.get('old_path')}"
    )
    choice = input("请输入 a=批准回退, r=拒绝并停止 [r]: " ).strip().lower()
    if choice != "a":
        print(f"已拒绝第 {restore_step_manual} 步回退，后续操作已停止。")
        break

    restore_result = parse_mcp_tool_result(
        await integrated_tool_by_name["file_restore_last_rename"].ainvoke(
            {"filter_dir": FILE_TOOL_DIRECTORY}
        )
    )
    restore_results_manual.append(restore_result)
    if not restore_result.get("ok"):
        print(f"❌ 第 {restore_step_manual} 步回退失败: {restore_result.get('message')}")
        break
    print(f"✅ 第 {restore_step_manual} 步回退成功: {restore_result.get('restored_path')}")

print(f"回退完成：计划 {len(restore_plan_manual)} 个，成功 {sum(r.get('ok', False) for r in restore_results_manual)} 个。")

files_after_restore_manual = parse_mcp_tool_result(
    await integrated_tool_by_name["file_list_files"].ainvoke({"directory": FILE_TOOL_DIRECTORY})
)
history_after_restore_manual = parse_mcp_tool_result(
    await integrated_tool_by_name["file_view_rename_history"].ainvoke({"filter_dir": FILE_TOOL_DIRECTORY})
)

files_after_restore_manual, history_after_restore_manual


## 12. 完整代码：一把执行综合 Agent

&emsp;&emsp;前面按步骤拆开讲解，是为了让每个环节都能单独观察。下面这个单元把第六章的完整流程合并到一起：启动 MCP、加载新闻 / 天气 / 文件工具、创建带 HITL 的综合 Agent、执行综合任务、在文件重命名和回退前弹出审批面板，并输出最终结果、审批记录、文件列表和重命名历史。

&emsp;&emsp;默认设置 `AUTO_APPROVE_FILE_CHANGES=False`、`USE_WIDGET_APPROVAL=False`，因此运行到 `file_rename_file` 或 `file_restore_last_rename` 时会使用 `input()` 进行审批，稳定性最好。如果希望在 notebook 中显示按钮式审批面板，可以把 `USE_WIDGET_APPROVAL` 改成 `True`；需要无人值守课堂演示时，可以把 `AUTO_APPROVE_FILE_CHANGES` 改成 `True`。


In [ ]:
print("【要点】一把执行版：串联 MCP 启动、工具加载、Agent 创建、HITL 审批和最终结果。")
# 一把执行版：新闻 + 天气 + 文件 Agent + Human-in-the-loop 审批
import asyncio
import getpass
import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

import httpx
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

AUTO_APPROVE_FILE_CHANGES = False  # True 表示无人值守自动批准；False 表示弹出审批面板。
USE_WIDGET_APPROVAL = False        # True: 用 ipywidgets 按钮审批；False: 用 input() 审批，最稳定。
WIDGET_APPROVAL_TIMEOUT = 8        # widget 前端卡住时，等待多少秒后降级到 input()。
VERBOSE_TOOL_RESULT = False        # True 时打印更长的工具返回内容。

LOG_WIDTH = 88


def line(char: str = "-"):
    print(char * LOG_WIDTH)


def section(title: str):
    print(f"\n{'=' * LOG_WIDTH}\n{title}\n{'=' * LOG_WIDTH}")


def subsection(title: str):
    print(f"\n{title}\n{'-' * LOG_WIDTH}")


def info(message: str):
    print(f"- {message}")


def short_text(value, limit: int = 260) -> str:
    text = json.dumps(value, ensure_ascii=False, default=str) if isinstance(value, (dict, list)) else str(value)
    text = " ".join(text.split())
    if len(text) > limit:
        return text[:limit] + "..."
    return text


def task_label(tool_name: str) -> str:
    if tool_name == "tavily_search":
        return "任务1 查新闻"
    if tool_name == "weather_get_weather":
        return "任务2 查天气"
    if tool_name == "file_list_files":
        return "任务3 列文件"
    if tool_name == "file_read_file":
        return "任务4 读文件"
    if tool_name in {"file_get_file_info", "file_list_allowed_roots"}:
        return "文件检查"
    if tool_name == "file_rename_file":
        return "任务5 重命名"
    if tool_name in {"file_view_rename_history", "file_restore_last_rename"}:
        return "任务7 回退"
    return "工具调用"


def summarize_tool_call(tool_call: dict) -> str:
    name = tool_call.get("name", "unknown_tool")
    args = tool_call.get("args", {})
    if name == "tavily_search":
        return f"query={args.get('query')}"
    if name == "weather_get_weather":
        return f"location={args.get('location')}"
    if name == "file_list_files":
        return f"directory={args.get('directory')}"
    if name == "file_read_file":
        return f"filepath={args.get('filepath')}"
    if name == "file_rename_file":
        return f"{args.get('old_path')} -> {args.get('new_name')}"
    if name == "file_restore_last_rename":
        return "恢复最近一次重命名"
    return short_text(args, limit=180)


def print_task_plan():
    subsection("任务清单")
    tasks = [
        "1. 查询一条最新的人工智能相关新闻",
        "2. 查询 Beijing 当前天气",
        "3. 列出 files/ 目录",
        "4. 逐个读取文件并判断编程语言和用途",
        "5. 重命名为合适英文文件名和代码后缀（需要审批）",
        "6. 总结新闻、天气、识别依据、重命名结果和回退方法",
        "7. 发起回退刚才的文件重命名（每次回退都需要审批，可拒绝）",
    ]
    for task in tasks:
        print(task)


def print_tool_trace(agent_result: dict, start_index: int = 0) -> int:
    """只打印新增消息中的工具调用轨迹，返回已打印到的消息数量。"""
    messages = agent_result.get("messages", [])
    new_messages = messages[start_index:]
    if not new_messages:
        return len(messages)

    printed_header = False
    for message in new_messages:
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            if not printed_header:
                subsection("工具调用轨迹")
                printed_header = True
            for call in tool_calls:
                name = call.get("name", "unknown_tool")
                print(f"[{task_label(name)}] {name}")
                print(f"  参数: {summarize_tool_call(call)}")

        msg_type = getattr(message, "type", "")
        if msg_type == "tool":
            if not printed_header:
                subsection("工具调用轨迹")
                printed_header = True
            name = getattr(message, "name", "unknown_tool")
            content = getattr(message, "content", "")
            limit = 800 if VERBOSE_TOOL_RESULT else 220
            print(f"  返回: {name} -> {short_text(content, limit=limit)}")
            line(".")

    if not printed_header:
        info("本轮没有新的工具调用。")
    return len(messages)


# 1. 加载环境变量与 API Key
section("阶段 1/7：环境准备")
load_dotenv(find_dotenv(usecwd=True), override=True)
# 一把执行版默认使用 vLLM Qwen 122B；切换 DeepSeek 时再配置 DEEPSEEK_API_KEY。
for env_name in ["vLLM_QWEN122B_API_KEY", "TAVILY_API_KEY", "OPENWEATHER_API_KEY"]:
    if not os.environ.get(env_name):
        os.environ[env_name] = getpass.getpass(f"{env_name}:\n")
    info(f"{env_name}: {'已配置' if os.environ.get(env_name) else '缺失'}")

for proxy_name in ["HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"]:
    os.environ.pop(proxy_name, None)
info("代理环境变量已清理。")

def resolve_one_shot_course_dir() -> Path:
    cwd = Path.cwd().resolve()
    relative_course_dir = Path("04_综合实战/06_综合实战-Langchain")
    for candidate in [cwd / relative_course_dir, cwd]:
        if (candidate / "mcp").is_dir() and (candidate / "files").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f"无法定位课程目录；cwd={cwd}")


COURSE_DIR = resolve_one_shot_course_dir()
WEATHER_MCP_SCRIPT = COURSE_DIR / "mcp/01_weather-mcp-server" / "openweather_mcp_server.py"
FILE_MCP_SCRIPT = COURSE_DIR / "mcp/02_file-mcp-server" / "secure_file_mcp_server.py"
DEFAULT_FILES_OPERATION_DIR = (COURSE_DIR / "files").resolve()
operation_dir_input = os.getenv("LANGCHAIN_FILES_OPERATION_DIR", "").strip()
if not operation_dir_input:
    operation_dir_input = input(
        f"请输入要操作的文件夹绝对路径 [{DEFAULT_FILES_OPERATION_DIR}]: "
    ).strip() or str(DEFAULT_FILES_OPERATION_DIR)
FILES_OPERATION_DIR = Path(operation_dir_input).expanduser()
if not FILES_OPERATION_DIR.is_absolute():
    raise ValueError(f"请输入绝对路径: {FILES_OPERATION_DIR}")
FILES_OPERATION_DIR = FILES_OPERATION_DIR.resolve()
if not FILES_OPERATION_DIR.is_dir():
    raise NotADirectoryError(f"文件操作目录不存在或不是目录: {FILES_OPERATION_DIR}")
FILE_TOOL_DIRECTORY = str(FILES_OPERATION_DIR)
ONE_SHOT_MCP_PROCESSES = {}
info(f"课程目录: {COURSE_DIR}")
info(f"文件操作目录: {FILES_OPERATION_DIR}")
info(f"目录存在: {FILES_OPERATION_DIR.is_dir()}")


MODEL_CONFIGS = {
    "deepseek-v4-flash": {
        "model": "deepseek-v4-flash",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的轻量快速模型",
    },
    "deepseek-v4-pro": {
        "model": "deepseek-v4-pro",
        "base_url": "https://api.deepseek.com",
        "api_key_env": "DEEPSEEK_API_KEY",
        "description": "DeepSeek 兼容 OpenAI API 的更强能力模型",
    },
    "vllm-qwen122b": {
        "model": os.getenv("vLLM_QWEN122B_MODEL_ID"),
        "base_url": os.getenv("vLLM_QWEN122B_BASE_URL"),
        "api_key_env": "vLLM_QWEN122B_API_KEY",
        "description": "vLLM 部署的 Qwen 122B 模型（默认）",
    },
    "Qwen/Qwen3-8B": {
        "model": "Qwen/Qwen3-8B",
        "base_url": "https://api.siliconflow.cn/v1",
        "api_key_env": "SILICONFLOW_API_KEY",
        "description": "SiliconFlow 上的 Qwen/Qwen3-8B 模型",
    },
    "qwen/qwen3-coder:free": {
        "model": "qwen/qwen3-coder:free",
        "base_url": "https://openrouter.ai/api/v1",
        "api_key_env": "OPENROUTER_API_KEY",
        "description": "OpenRouter 上的 Qwen3 Coder 免费模型",
    },
}


def create_model(model_name: str = "vllm-qwen122b", temperature: float = 0.1, max_tokens: int = 10000):
    """创建一把执行版使用的模型；默认 vllm-qwen122b。切换 DeepSeek：create_model("deepseek-v4-flash")。"""
    if model_name not in MODEL_CONFIGS:
        raise ValueError(f"不支持的 model_name: {model_name}。可选值: {', '.join(MODEL_CONFIGS)}")
    config = MODEL_CONFIGS[model_name]
    if not config.get("model") or not config.get("base_url"):
        raise ValueError(f"模型 {model_name} 缺少 model 或 base_url 配置，请检查 .env。")
    api_key = os.getenv(config["api_key_env"])
    if not api_key:
        raise ValueError(f"缺少环境变量 {config['api_key_env']}，请先在 .env 中配置。")
    info(f"模型: {model_name} | {config['description']} | temperature={temperature}, max_tokens={max_tokens}")
    return init_chat_model(
        model=config["model"],
        model_provider="openai",
        base_url=config["base_url"],
        api_key=api_key,
        http_client=httpx.Client(trust_env=False),
        temperature=temperature,
        max_tokens=max_tokens,
    )


def is_one_shot_port_open(port: int, host: str = "127.0.0.1", timeout: float = 0.5) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def start_one_shot_mcp_server(
    name: str, script_path: Path, port: int, extra_env: dict | None = None, reuse_existing: bool = True
):
    """启动本地 MCP Server；可选是否复用已占用端口的服务。"""
    if is_one_shot_port_open(port):
        if reuse_existing:
            info(f"{name} MCP: 复用已有服务，端口 {port}")
            return {"name": name, "port": port, "status": "already_running"}
        raise RuntimeError(
            f"{name} MCP 端口 {port} 已被占用，无法确认正在运行的是本章服务。"
            "请先停止占用该端口的旧服务。"
        )

    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)

    process = subprocess.Popen(
        [sys.executable, str(script_path)],
        cwd=str(script_path.parent),
        env=env,
        # 不使用未持续消费的 PIPE，避免 MCP 日志写满缓冲区后阻塞。
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        text=True,
    )

    for _ in range(40):
        if is_one_shot_port_open(port):
            ONE_SHOT_MCP_PROCESSES[name] = process
            info(f"{name} MCP: 已启动，端口 {port}, pid={process.pid}")
            return {"name": name, "port": port, "status": "started", "pid": process.pid}
        if process.poll() is not None:
            stderr = process.stderr.read() if process.stderr else ""
            raise RuntimeError(f"{name} 启动失败：{stderr}")
        time.sleep(0.25)

    process.terminate()
    raise TimeoutError(f"{name} 在端口 {port} 上启动超时。")


def parse_one_shot_mcp_tool_result(result):
    """把 langchain-mcp-adapters 返回的 content blocks 尽量解析为 Python 对象。"""
    if isinstance(result, list) and result and isinstance(result[0], dict):
        text = result[0].get("text", "")
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {"raw_text": text}
    return result


def describe_one_shot_file_modification(tool_call, state, runtime) -> str:
    """为文件修改类工具生成审批说明。"""
    tool_name = tool_call["name"]
    tool_args = tool_call.get("args", {})

    if tool_name == "file_rename_file":
        return (
            "准备执行文件重命名，请人工确认：\n"
            f"- old_path: {tool_args.get('old_path')}\n"
            f"- new_name: {tool_args.get('new_name')}\n"
            f"- base_dir: {tool_args.get('base_dir') or '(默认允许根目录)'}\n"
            "请确认该操作只作用于 files/ 样例目录，且新文件名和后缀符合教学目标。"
        )

    if tool_name == "file_restore_last_rename":
        return (
            "准备回退最近一次文件重命名，请人工确认：\n"
            f"- filter_dir: {tool_args.get('filter_dir') or '(all history)'}\n"
            "- 该操作会把最近一次 rename 历史恢复到旧文件名。\n"
            "- 如果需要连续回退多个文件，Agent 可能会多次请求审批。"
        )

    return f"准备执行文件修改工具 {tool_name}，参数：{tool_args}"


def get_one_shot_hitl_action_requests(agent_result: dict) -> list[dict]:
    """从 Agent 中断结果中提取待人工审批的工具动作。"""
    interrupts = agent_result.get("__interrupt__", ())
    if not interrupts:
        return []
    hitl_request = interrupts[0].value
    return hitl_request.get("action_requests", [])


def print_one_shot_hitl_actions(actions: list[dict]):
    """在终端输出待审批动作摘要，详细参数在弹窗中展示。"""
    if not actions:
        info("没有待审批的文件修改动作。")
        return
    info(f"待审批动作数: {len(actions)}")
    for index, action in enumerate(actions, start=1):
        print(f"  {index}. {summarize_tool_call(action)}")


def build_approval_decision(action: dict, edited_args: dict | None = None) -> dict:
    """根据是否修改参数生成 HITL decision。"""
    name = action.get("name")
    original_args = action.get("args", {})
    if edited_args is None or edited_args == original_args:
        return {"type": "approve"}
    return {"type": "edit", "edited_action": {"name": name, "args": edited_args}}


async def approve_actions_with_widgets(actions: list[dict]) -> list[dict] | None:
    """在 Jupyter 中显示审批面板；如果 ipywidgets 不可用，返回 None。"""
    if not USE_WIDGET_APPROVAL:
        return None

    try:
        import ipywidgets as widgets
        from IPython.display import display
    except Exception as exc:
        info(f"ipywidgets 不可用，改用命令行审批。原因：{exc}")
        return None

    decisions = []
    loop = asyncio.get_running_loop()

    for index, action in enumerate(actions, start=1):
        future = loop.create_future()
        name = action.get("name", "")
        args = dict(action.get("args", {}))
        description = action.get("description", "(无审批说明)")

        title = widgets.HTML(
            value=(
                f"<h3 style='margin:0 0 8px 0'>文件修改审批 {index}/{len(actions)}</h3>"
                f"<p><b>工具：</b><code>{name}</code></p>"
                f"<p><b>摘要：</b>{summarize_tool_call(action)}</p>"
            )
        )
        desc = widgets.HTML(
            value="<pre style='white-space:pre-wrap;background:#f7f7f7;padding:8px;border:1px solid #ddd'>"
            + description
            + "</pre>"
        )

        arg_widgets = {}
        arg_rows = []
        for key, value in args.items():
            widget = widgets.Text(
                value="" if value is None else str(value),
                description=key,
                layout=widgets.Layout(width="95%"),
                style={"description_width": "120px"},
            )
            arg_widgets[key] = widget
            arg_rows.append(widget)

        if not arg_rows:
            arg_rows.append(widgets.HTML(value="<p>该工具没有参数可编辑。</p>"))

        status = widgets.HTML(value="<span style='color:#666'>请选择审批结果。</span>")
        approve_button = widgets.Button(description="批准", button_style="success", icon="check")
        edit_button = widgets.Button(description="按当前参数批准", button_style="warning", icon="edit")
        reject_button = widgets.Button(description="拒绝", button_style="danger", icon="times")

        def current_args():
            edited = dict(args)
            for key, widget in arg_widgets.items():
                edited[key] = widget.value
            return edited

        def finish(decision):
            if not future.done():
                future.set_result(decision)

        def on_approve(_):
            status.value = "<b style='color:green'>已批准，继续执行。</b>"
            finish({"type": "approve"})

        def on_edit(_):
            edited = current_args()
            status.value = "<b style='color:#996600'>已按当前参数批准，继续执行。</b>"
            finish(build_approval_decision(action, edited))

        def on_reject(_):
            status.value = "<b style='color:red'>已拒绝，该工具不会执行。</b>"
            finish({"type": "reject", "message": f"人工审批拒绝执行 {name}。"})

        approve_button.on_click(on_approve)
        edit_button.on_click(on_edit)
        reject_button.on_click(on_reject)

        buttons = widgets.HBox([approve_button, edit_button, reject_button])
        panel = widgets.VBox(
            [title, desc, *arg_rows, buttons, status],
            layout=widgets.Layout(border="2px solid #999", padding="12px", margin="8px 0"),
        )
        display(panel)
        info(f"如果只看到 `Loading widget...`，{WIDGET_APPROVAL_TIMEOUT} 秒后会自动切到 input() 审批。")

        try:
            decision = await asyncio.wait_for(future, timeout=WIDGET_APPROVAL_TIMEOUT)
        except asyncio.TimeoutError:
            panel.close()
            info("widget 审批面板未响应，降级为 input() 审批。")
            return None

        decisions.append(decision)
        panel.close()

    return decisions


def approve_actions_from_terminal(actions: list[dict]) -> list[dict]:
    """非 Jupyter 前端环境下的命令行审批兜底。"""
    decisions = []
    for index, action in enumerate(actions, start=1):
        print(f"\n审批 {index}/{len(actions)}: {summarize_tool_call(action)}")
        choice = input("请输入 a=批准, r=拒绝 [r]: ").strip().lower()
        if choice == "a":
            decisions.append({"type": "approve"})
        else:
            decisions.append({"type": "reject", "message": f"人工审批拒绝执行 {action.get('name')}。"})
    return decisions


async def review_one_shot_actions(actions: list[dict]) -> list[dict]:
    """统一审批入口：自动审批、Jupyter 弹窗审批、命令行审批三选一。"""
    if AUTO_APPROVE_FILE_CHANGES:
        info("AUTO_APPROVE_FILE_CHANGES=True，自动批准所有文件修改动作。")
        return [{"type": "approve"} for _ in actions]

    widget_decisions = await approve_actions_with_widgets(actions)
    if widget_decisions is not None:
        return widget_decisions

    return approve_actions_from_terminal(actions)


def print_decision_summary(decisions: list[dict]):
    counts = {}
    for decision in decisions:
        key = decision.get("type", "unknown")
        counts[key] = counts.get(key, 0) + 1
    info("审批结果: " + ", ".join(f"{k}={v}" for k, v in counts.items()))


# 2. 启动天气 MCP 与文件 MCP
section("阶段 2/7：启动 MCP 服务")
info("文件 MCP 不设允许根目录")
info(f"本章指定操作目录: {FILES_OPERATION_DIR}")
one_shot_weather_status = start_one_shot_mcp_server(
    name="weather",
    script_path=WEATHER_MCP_SCRIPT,
    port=16681,
    extra_env={"WEATHER_MCP_PORT": "16681"},
)
one_shot_file_status = start_one_shot_mcp_server(
    name="file",
    script_path=FILE_MCP_SCRIPT,
    port=16682,
    reuse_existing=(
        "INTEGRATED_MCP_PROCESSES" in globals()
        and "file" in globals()["INTEGRATED_MCP_PROCESSES"]
    ),
    extra_env={
        "FILE_MCP_MAX_READ_BYTES": "200000",
    },
)

# 3. 加载新闻、天气和文件工具
section("阶段 3/7：加载工具")
one_shot_news_search = TavilySearch(max_results=3, topic="news")
one_shot_mcp_client = MultiServerMCPClient(
    {
        "weather": {"url": "http://127.0.0.1:16681/mcp", "transport": "http"},
        "file": {"url": "http://127.0.0.1:16682/mcp", "transport": "http"},
    },
    tool_name_prefix=True,
)
one_shot_mcp_tools = await one_shot_mcp_client.get_tools()
one_shot_tools = [one_shot_news_search, *one_shot_mcp_tools]
one_shot_tool_by_name = {tool.name: tool for tool in one_shot_tools}
info(f"工具总数: {len(one_shot_tools)}")
info("关键工具: tavily_search / weather_get_weather / file_list_files / file_read_file / file_rename_file / file_restore_last_rename")

# 4. 创建带记忆和 HITL 审批的综合 Agent
section("阶段 4/7：创建 Agent")
one_shot_agent = create_agent(
    # 默认使用 vllm-qwen122b；切换 DeepSeek 可改为 create_model("deepseek-v4-flash")。
    model=create_model(),
    tools=one_shot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "file_rename_file": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "description": describe_one_shot_file_modification,
                },
                "file_restore_last_rename": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": describe_one_shot_file_modification,
                },
            },
            description_prefix="文件修改操作需要人工审批",
        )
    ],
    system_prompt=(
        "你是一个面向课程实战的综合 Agent，可以查新闻、查天气、操作课程 files 目录中的样例文件。"
        "当用户询问最新新闻、近期动态或需要外部事实核实时，优先调用 tavily_search。"
        "当用户询问天气时，优先调用 weather_get_weather，参数名使用 location，城市名使用英文，例如 Beijing、Shanghai。"
        "当用户要求查看文件时，使用 file_list_files、file_read_file、file_get_file_info 等文件 MCP 工具。"
        f"文件 MCP 是通用服务；用户本次指定的操作目录是 {FILES_OPERATION_DIR}。"
        f"调用文件工具时，directory 和 filter_dir 使用 {FILE_TOOL_DIRECTORY}，filepath/old_path 使用该目录下的绝对路径。"
        f"调用 file_restore_last_rename 时必须传入 filter_dir={FILE_TOOL_DIRECTORY}，只回退当前选定目录的历史。"
        "只有用户明确授权整理、重命名或回退文件时，才允许调用 file_rename_file 或 file_restore_last_rename。"
        "重命名时必须先读取文件内容，根据内容判断语言和用途，再生成英文小写、下划线分隔、后缀正确的新文件名。"
        "执行文件重命名或回退前，运行时会要求人工审批；如果审批被拒绝，不要重复调用同一个修改工具。"
        "回答时用中文说明调用了哪些工具、识别依据、重命名结果，以及如何回退。"
    ),
)
one_shot_agent_config = {
    "configurable": {"thread_id": f"one-shot-news-weather-file-agent-{int(time.time())}"}
}
info(f"thread_id: {one_shot_agent_config['configurable']['thread_id']}")

# 5. 执行综合任务；遇到文件修改工具时会触发 HITL 中断。
section("阶段 5/7：执行综合任务")
print_task_plan()
one_shot_result = await one_shot_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "请完成一个综合实战任务，我明确授权你整理 files/ 目录中的样例文件："
                    "1）查询一条最新的人工智能相关新闻；"
                    "2）查询 Beijing 当前天气；"
                    "3）列出 files/ 目录，逐个读取里面的文件；"
                    "4）根据每个文件内容判断编程语言和用途；"
                    "5）把这些文件重命名为合适的英文文件名，并使用正确的代码文件后缀，例如 .py、.go、.java、.c、.js；"
                    "6）用中文总结新闻、天气、文件识别依据、重命名结果和回退方法；"
                    f"7）总结后，请继续发起回退，调用 file_restore_last_rename 时每次传入 filter_dir={FILE_TOOL_DIRECTORY}；"
                    "每一次回退都会触发人工审批，审批人可以批准或拒绝。"
                    "如果审批被拒绝，请停止继续回退，并说明当前保留的文件状态。"
                ),
            }
        ]
    },
    config=one_shot_agent_config,
)
printed_message_count = print_tool_trace(one_shot_result, 0)

# 6. 文件修改请求触发 HITL，中断后弹出审批面板并继续执行。
section("阶段 6/7：处理人工审批")
approved_file_actions = []
approval_decisions_all = []
approval_round = 0
while "__interrupt__" in one_shot_result:
    approval_round += 1
    pending_actions = get_one_shot_hitl_action_requests(one_shot_result)
    approved_file_actions.extend(pending_actions)

    subsection(f"审批轮次 {approval_round}")
    print_one_shot_hitl_actions(pending_actions)

    approval_decisions = await review_one_shot_actions(pending_actions)
    approval_decisions_all.extend(approval_decisions)
    print_decision_summary(approval_decisions)

    one_shot_result = await one_shot_agent.ainvoke(
        Command(resume={"decisions": approval_decisions}),
        config=one_shot_agent_config,
    )
    printed_message_count = print_tool_trace(one_shot_result, printed_message_count)

info(f"审批结束: {approval_round} 轮，{len(approved_file_actions)} 个文件修改动作")

# 7. 查看执行后的文件列表和重命名历史。
section("阶段 7/7：最终结果")
one_shot_files_listing = parse_one_shot_mcp_tool_result(
    await one_shot_tool_by_name["file_list_files"].ainvoke({"directory": FILE_TOOL_DIRECTORY})
)
one_shot_rename_history = parse_one_shot_mcp_tool_result(
    await one_shot_tool_by_name["file_view_rename_history"].ainvoke({"filter_dir": FILE_TOOL_DIRECTORY})
)
file_names = [item.get("name") for item in one_shot_files_listing.get("items", []) if item.get("is_file")]
final_answer = one_shot_result["messages"][-1].content
info(f"操作目录: {FILES_OPERATION_DIR}")
info(f"当前文件: {', '.join(file_names) if file_names else '无文件'}")
info(f"重命名历史条数: {one_shot_rename_history.get('count', 0)}")
print("\n最终回答:\n" + final_answer)

{
    "mcp_status": {
        "weather": one_shot_weather_status,
        "file": one_shot_file_status,
    },
    "approved_file_actions": approved_file_actions,
    "approval_decisions": approval_decisions_all,
    "final_answer": final_answer,
    "files_listing": one_shot_files_listing,
    "rename_history": one_shot_rename_history,
}


## 13. 关闭本章启动的 MCP 服务

&emsp;&emsp;如果服务是由本章启动的，运行下面单元可以释放进程。如果端口原本已经有外部服务在运行，本单元不会关闭外部服务。


In [ ]:
print("【要点】课程结束后关闭本 notebook 启动的 MCP 子进程，释放端口。")
for process_dict_name in ["INTEGRATED_MCP_PROCESSES", "ONE_SHOT_MCP_PROCESSES"]:
    process_dict = globals().get(process_dict_name, {})
    for name, process in list(process_dict.items()):
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
        print(f"{name} MCP server stopped from {process_dict_name}")
    process_dict.clear()


## 14. 本章小结

&emsp;&emsp;到这里，我们已经把前面章节的关键元素组合成一个完整 Agent，并按照“规划 → 执行 → 查看 → 回退”的顺序完成了一次文件整理实战：

- **模型**：使用 `create_model()` 统一接入 `deepseek-v4-flash`、`deepseek-v4-pro`、`Qwen/Qwen3-8B` 或 `qwen/qwen3-coder:free`。
- **工具**：使用 `TavilySearch` 查新闻，使用 MCP 工具查天气和操作文件。
- **MCP**：通过 `MultiServerMCPClient` 将独立 MCP Server 转换为 LangChain Tool，并用工具名前缀区分天气服务和文件服务。
- **ReAct**：Agent 根据任务自动选择工具、观察结果并继续推理，完成新闻、天气、文件读取和文件整理规划。
- **记忆**：通过 `InMemorySaver` 和 `thread_id` 保留多轮上下文，让后续问题可以复用前面形成的文件理解结果。
- **文件理解与写入**：通过 `file_read_file` 读取内容，由 Agent 判断编程语言和用途，再通过 `file_rename_file` 把整理结果写回到文件名和后缀上。
- **结果可追踪**：通过 `file_list_files` 查看当前文件列表，通过 `file_view_rename_history` 查看每次重命名留下的历史记录。
- **Human-in-the-loop 审批**：通过 `HumanInTheLoopMiddleware` 拦截 `file_rename_file` 和 `file_restore_last_rename`，确保文件修改动作在执行前经过自动审批或人工确认。
- **可回退设计**：通过 rename history 生成明确的回退计划，打印需要回退的文件数量，以及每一步 `new_path -> old_path` 的恢复方向，再调用 `file_restore_last_rename` 逐条恢复。
- **安全约束**：只允许在 `files/` 样例目录内进行文件整理，避免误操作课程源码或 notebook。

&emsp;&emsp;真实项目中，可以继续把这里的内存型 checkpointer 换成数据库持久化，把本地 MCP 服务部署成长期运行服务，并把审批界面接到企业内部工单、IM 或管理后台中。


# 第七章 本文档涉及到的 API 清单与解析

&emsp;&emsp;本章把前面章节中出现的核心 API 统一梳理一遍。学习时可以把它当作速查表：先看“作用”，再看“关键参数 / 返回值”，最后回到前面的代码单元中对照运行。

## 1. 环境变量与配置加载

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `load_dotenv()` | `python-dotenv` | 从 `.env` 文件加载环境变量 | `load_dotenv(override=True)` 会用 `.env` 中的值覆盖当前进程已有环境变量；`load_dotenv(env_path)` 可以指定 `.env` 路径。本文档用它加载 `DEEPSEEK_API_KEY`、`SILICONFLOW_API_KEY`、`OPENROUTER_API_KEY`、`TAVILY_API_KEY`、`OPENWEATHER_API_KEY` 等。 |
| `os.getenv()` | Python 标准库 | 读取环境变量 | `os.getenv("SILICONFLOW_API_KEY")` 返回字符串或 `None`。适合读取 API Key，不建议把 Key 直接写进代码。 |
| `getpass.getpass()` | Python 标准库 | 在命令行隐藏输入敏感信息 | 当环境变量不存在时，用它临时输入 `TAVILY_API_KEY` 或其他 Key，输入内容不会明文显示。 |
| `httpx.Client(trust_env=False)` | `httpx` | 创建 HTTP 客户端 | `trust_env=False` 表示不读取系统代理环境变量。本文档用于避免本机代理影响 DeepSeek 或 SiliconFlow 的 OpenAI 兼容接口请求。 |

## 2. 模型接入与基础调用 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `OpenAI()` | `openai` | 创建 OpenAI SDK 客户端 | `OpenAI(api_key=..., base_url=...)` 可连接 OpenAI 兼容服务。示例中通过 `base_url="https://api.deepseek.com"` 接入 DeepSeek。 |
| `client.chat.completions.create()` | `openai` | 使用 OpenAI Chat Completions 接口发起模型调用 | 常用参数包括 `model`、`messages`。返回对象中可通过 `response.choices[0].message.content` 取最终文本。 |
| `ChatDeepSeek()` | `langchain-deepseek` | LangChain 封装的 DeepSeek 聊天模型 | `ChatDeepSeek(model="deepseek-chat")` 创建普通对话模型；`deepseek-reasoner` 可返回推理相关扩展信息。 |
| `init_chat_model()` | `langchain.chat_models` | LangChain 统一模型初始化入口 | `create_model()` 底层使用它接入 OpenAI 兼容模型服务；本文档示例覆盖 DeepSeek 的 `deepseek-v4-flash` / `deepseek-v4-pro`、SiliconFlow 的 `Qwen/Qwen3-8B` 和 OpenRouter 的 `qwen/qwen3-coder:free`。 |
| `model.invoke()` | LangChain Chat Model | 同步调用模型或 Agent | 输入可以是字符串、消息列表，或 Agent 使用的 `{"messages": ...}` 字典；返回通常是 `AIMessage` 或 Agent 状态字典。 |
| `model.stream()` | LangChain Chat Model | 流式生成模型回复 | 返回一个可迭代对象，每次产出一个增量 `chunk`。适合终端逐字输出或 Gradio 流式聊天。 |
| `model.batch()` | LangChain Chat Model | 批量并发调用 | 输入多个问题组成的列表，返回结果列表。适合多个彼此独立的问题。 |
| `model.batch_as_completed()` | LangChain Chat Model | 按完成顺序返回批量调用结果 | 哪个请求先完成就先返回哪个结果，适合任务耗时不均的批处理场景。 |
| `model.with_structured_output()` | LangChain Chat Model | 让模型按指定结构返回结果 | 通常传入 Pydantic 模型，例如 `Movie`。OpenAI 兼容模型服务建议显式设置 `method="function_calling"`；`include_raw=True` 会同时返回原始模型响应，便于调试结构化解析失败的问题。 |

## 3. 消息对象 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `SystemMessage` | `langchain.messages` / `langchain_core.messages` | 表示系统提示词 | 用来设定助手身份、回答风格、工具使用规则和安全边界，通常放在消息列表最前面。 |
| `HumanMessage` | `langchain.messages` / `langchain_core.messages` | 表示用户消息 | 可直接构造 `HumanMessage(content="...")`，也可在 Agent 输入中使用 `{"role": "user", "content": "..."}` 的字典形式。 |
| `AIMessage` | `langchain.messages` / `langchain_core.messages` | 表示模型回复 | 多轮对话中把模型回复追加回 `messages`，模型才能在下一轮看到历史上下文。 |
| `RemoveMessage` | `langchain.messages` | 从 LangGraph 状态中删除消息 | 常与中间件配合使用，用于裁剪长上下文。可以删除指定消息 ID，也可以结合 `REMOVE_ALL_MESSAGES` 先清空再写回保留消息。 |
| `REMOVE_ALL_MESSAGES` | `langgraph.graph.message` | 删除全部历史消息的特殊标记 | 在 `before_model` 中用于“整体替换消息列表”，比逐条删除更适合上下文裁剪。 |

## 4. Agent 创建、运行与状态 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `create_agent()` | `langchain.agents` | 创建 LangChain 1.0 Agent | 核心参数包括 `model`、`tools`、`system_prompt`、`middleware`、`checkpointer`。底层由 LangGraph Runtime 管理模型调用、工具调用和状态流转。 |
| `agent.invoke()` | LangChain Agent | 同步运行 Agent | 输入通常是 `{"messages": [{"role": "user", "content": "..."}]}`。返回字典中 `result["messages"][-1].content` 是最终回答。 |
| `agent.ainvoke()` | LangChain Agent | 异步运行 Agent | 在 notebook 中可直接使用 `await agent.ainvoke(...)`。适合 MCP 工具、网络工具等异步链路。 |
| `agent.get_state()` | LangGraph Agent | 查看指定配置下的 Agent 状态 | 需要传入包含 `thread_id` 的 `config`，可用于教学中观察记忆、消息和运行状态。 |
| `RunnableConfig` | `langchain_core.runnables` | LangChain / LangGraph 运行配置类型 | 示例中主要用 `{"configurable": {"thread_id": "..."}}` 区分会话。相同 `thread_id` 会复用同一条记忆状态。 |
| `Command()` | `langgraph.types` | 恢复被中断的 LangGraph / Agent 运行 | HITL 审批后使用 `Command(resume={"decisions": [...]})` 把人工决策传回 Agent，让暂停的工具调用继续、修改或拒绝执行。 |
| `AgentState` | `langchain.agents` | Agent 状态类型 | 中间件函数通过它读取或修改 `state["messages"]` 等运行时状态。 |

## 5. 工具 Tool API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `TavilySearch()` | `langchain_tavily` | 创建 Tavily 搜索工具 | `max_results` 控制返回条数；`topic="general"` 用于通用搜索，`topic="news"` 用于新闻搜索。创建后可传入 `create_agent(tools=[...])`。 |
| `tool.invoke()` | LangChain Tool | 直接同步调用工具 | 示例中 `web_search.invoke({"query": "..."})` 用来先验证工具链路，再交给 Agent 使用。 |
| `tool.ainvoke()` | LangChain Tool | 直接异步调用工具 | MCP 工具多用 `await tool.ainvoke({...})`，适合 notebook 中验证天气、文件读取、文件重命名等工具。 |
| `@tool` | `langchain.tools` | 把普通 Python 函数包装成 LangChain Tool | 函数名会成为工具名，函数 docstring 会成为工具描述，类型注解和参数名会形成工具 schema，供模型决定如何调用。 |
| `tool.name` | LangChain Tool 属性 | 查看工具名称 | Agent 工具调用时使用该名称。MCP 工具开启前缀后会出现 `weather_get_weather`、`file_read_file` 等名称。 |
| `tool.description` | LangChain Tool 属性 | 查看工具描述 | 描述越清晰，模型越容易选对工具。本文档通过打印它帮助理解 `@tool` 的包装结果。 |
| `tool.args` | LangChain Tool 属性 | 查看工具参数 schema | 可以看到工具需要哪些参数、参数类型是什么。用于调试模型传参是否符合工具要求。 |

## 6. MCP 服务与 MCP 工具 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `MultiServerMCPClient()` | `langchain_mcp_adapters.client` | 连接一个或多个 MCP Server | 传入服务名到连接配置的映射，例如 `{"weather": {"url": "http://127.0.0.1:16681/mcp", "transport": "http"}}`。`tool_name_prefix=True` 可避免多个服务工具重名。 |
| `mcp_client.get_tools()` | `langchain-mcp-adapters` | 把 MCP Server 暴露的工具加载为 LangChain Tool | 返回工具列表，之后可直接放入 `create_agent(tools=mcp_tools)`。这是 MCP 与 LangChain Agent 连接的关键步骤。 |
| `weather_ping` / `file_ping` | 本课程 MCP 工具 | 健康检查 | 用来确认 MCP Server 是否可连接。开启 `tool_name_prefix=True` 后，工具名前带服务名前缀。 |
| `weather_check_config` | 本课程天气 MCP 工具 | 检查天气服务配置 | 用于确认 `OPENWEATHER_API_KEY` 是否配置。适合在调用真实天气查询前做诊断。 |
| `weather_get_weather` | 本课程天气 MCP 工具 | 查询实时天气 | 参数通常是 `{"location": "Beijing"}`。内部调用 OpenWeather API，返回城市、温度、天气描述等信息。 |
| `file_list_allowed_roots` | 本课程文件 MCP 工具 | 查看允许访问的根目录 | 用于确认文件 MCP 的安全边界，避免 Agent 访问课程目录之外的文件。 |
| `file_list_files` | 本课程文件 MCP 工具 | 列出目录内容 | 参数示例：`{"directory": "files"}`。返回文件名、大小、是否为文件等元数据。 |
| `file_read_file` | 本课程文件 MCP 工具 | 读取文件内容 | 参数示例：`{"filepath": "files/a.txt"}`。综合实战中用它读取样例文件并让 Agent 判断语言。 |
| `file_get_file_info` | 本课程文件 MCP 工具 | 查看单个文件信息 | 用于获取路径、大小、修改时间等元数据，适合在执行文件操作前做检查。 |
| `file_rename_file` | 本课程文件 MCP 工具 | 在允许范围内重命名文件 | 综合实战中由 Agent 在用户明确授权后调用。应限制在 `files/` 目录内，并生成后缀正确的新文件名。 |
| `file_view_rename_history` | 本课程文件 MCP 工具 | 查看重命名历史 | 参数可使用 `{"filter_dir": "files"}` 只看样例目录历史。用于确认哪些文件被改名。 |
| `file_restore_last_rename` | 本课程文件 MCP 工具 | 回退最近一次重命名 | 每次只恢复一条历史记录。本文档用循环多次调用，逐步回退文件整理结果。 |
| `parse_mcp_tool_result()` | 本 notebook 辅助函数 | 解析 MCP 工具返回内容 | `langchain-mcp-adapters` 的结果常是 content blocks；该函数尝试取出 `text` 并用 `json.loads()` 转成 Python 对象。 |

## 7. 记忆管理 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `InMemorySaver()` | `langgraph.checkpoint.memory` | 内存型 checkpointer | 让 Agent 在同一个 `thread_id` 下保存消息状态。适合课程演示，不适合生产长期持久化。 |
| `checkpointer` 参数 | `create_agent()` | 接入记忆存储 | `create_agent(..., checkpointer=InMemorySaver())` 后，运行时必须传入带 `thread_id` 的 `config` 才能区分不同会话。 |
| `thread_id` | LangGraph configurable 配置 | 标识一条会话线程 | 相同 `thread_id` 共享历史，不同 `thread_id` 互相隔离。本文档用它演示“记得用户姓名”和综合 Agent 的上下文延续。 |

## 8. 中间件 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `@wrap_model_call` | `langchain.agents.middleware` | 包装模型调用过程 | 可在模型真正调用前修改 `request.model`，也可记录日志、做路由或注入策略。示例中根据问题复杂度动态选择模型参数。 |
| `ModelRequest` | `langchain.agents.middleware` | 模型调用请求对象 | 包含当前模型、状态和上下文。中间件可通过 `request.state.get("messages", [])` 读取消息历史。 |
| `ModelResponse` | `langchain.agents.middleware` | 模型调用响应对象 | `wrap_model_call` 包装函数的返回类型。通常直接返回 `handler(request)` 的结果。 |
| `@before_model` | `langchain.agents.middleware` | 在模型调用前执行逻辑 | 适合做消息裁剪、上下文注入、权限检查等。返回 `dict` 可更新 Agent 状态；返回 `None` 表示不修改。 |
| `@after_model` | `langchain.agents.middleware` | 在模型调用后执行逻辑 | 适合做消息清理、日志记录、状态压缩等。示例中用它删除较早消息。 |
| `SummarizationMiddleware()` | `langchain.agents.middleware` | 自动摘要长对话 | `max_tokens_before_summary` 控制触发阈值，`messages_to_keep` 控制摘要后保留的原始消息数量，需要提供摘要模型。 |
| `HumanInTheLoopMiddleware()` | `langchain.agents.middleware` | 工具执行前加入人工审批 | `interrupt_on` 指定需要拦截的工具；`allowed_decisions` 可设置 `approve`、`edit`、`reject`、`respond`。第六章用它拦截 `file_rename_file` 和 `file_restore_last_rename`。 |

## 9. 外部 HTTP 与本地服务 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `requests.get()` | `requests` | 发起同步 HTTP GET 请求 | 自定义天气函数中用它调用 `https://api.openweathermap.org/data/2.5/weather`。`params` 中包含城市、API Key、单位和语言。 |
| `response.json()` | `requests` | 把 HTTP 响应解析成 JSON 对象 | OpenWeather 返回 JSON 后，示例再用 `json.dumps()` 转成字符串，便于作为工具结果返回给模型。 |
| `subprocess.Popen()` | Python 标准库 | 启动本地 MCP Server 子进程 | 本文档用它从 notebook 中启动天气和文件 MCP 服务，并把环境变量传给服务进程。 |
| `socket.socket()` / `connect_ex()` | Python 标准库 | 检查本地端口是否已打开 | `is_port_open()` 用它判断 `16681`、`16682` 端口是否已有 MCP 服务，避免重复启动。 |
| `Path.cwd()` | `pathlib` | 获取当前 notebook 工作目录 | 用来定位 `mcp/`、`files/` 等相对路径，并配置文件 MCP 的允许根目录。 |

## 10. Gradio 聊天界面 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `gr.Blocks()` | `gradio` | 创建可组合的 Web UI | 示例中用它组织聊天机器人界面，包括标题、聊天窗口、输入框和按钮。 |
| `gr.Chatbot()` | `gradio` | 聊天消息展示组件 | 接收 `(user, assistant)` 格式的历史列表，支持复制按钮和头像配置。 |
| `gr.Textbox()` | `gradio` | 用户输入框 | 绑定 `submit` 事件后，回车即可触发响应函数。 |
| `gr.Button()` | `gradio` | 按钮组件 | 示例中包含“发送”和“清空”按钮，分别绑定回复函数和清空函数。 |
| `gr.State()` | `gradio` | 保存会话状态 | 用来保存 LangChain `messages` 列表，使 Web UI 中的多轮对话能延续上下文。 |
| `demo.launch()` | `gradio` | 启动 Web 应用 | `server_name="0.0.0.0"` 允许外部访问，`server_port=7860` 指定端口，`debug=True` 便于教学调试。 |

## 11. Pydantic 结构化输出 API

| API | 来源 | 作用 | 关键用法与解析 |
| --- | --- | --- | --- |
| `BaseModel` | `pydantic` | 定义结构化数据模型 | 示例中 `Movie(BaseModel)` 定义了电影标题、年份、导演和评分字段，作为模型结构化输出的目标格式。 |
| `Field()` | `pydantic` | 给字段添加约束和说明 | `Field(..., description="...")` 中的 `...` 表示必填；`description` 会帮助模型理解字段含义。 |

## 12. 本课程封装函数与对象速查

| 名称 | 类型 | 作用 | 解析 |
| --- | --- | --- | --- |
| `create_model()` | 本 notebook 函数 | 统一创建 OpenAI 兼容聊天模型 | 支持 `deepseek-v4-flash`、`deepseek-v4-pro`、`Qwen/Qwen3-8B` 和 `qwen/qwen3-coder:free` 四个选择，自动匹配 `base_url` 与 API Key 环境变量。可通过 `temperature`、`max_tokens` 调整生成风格和最大输出长度。 |
| `get_weather()` | 自定义函数 / LangChain Tool | 查询城市天气 | 先作为普通函数调用 OpenWeather，后通过 `@tool` 包装给 Agent 使用，用来对比“普通函数”和“工具”的区别。 |
| `is_port_open()` | 本 notebook 函数 | 检查 MCP 服务端口 | 用于启动 MCP 前判断端口是否已被占用。 |
| `start_mcp_server()` | 本 notebook 函数 | 启动本地 MCP 服务 | 封装 `subprocess.Popen()`、端口等待、错误处理和进程记录。 |
| `dynamic_model_routing()` | 中间件函数 | 动态模型路由 | 根据消息数量、用户输入长度和复杂任务关键词，在普通配置与稳健配置模型之间切换。 |
| `trim_messages()` | 中间件函数 | 模型调用前裁剪消息 | 超过阈值时保留首条消息和最近几条消息，降低上下文长度。 |
| `delete_old_messages()` | 中间件函数 | 模型调用后删除旧消息 | 演示如何在 `after_model` 阶段修改 LangGraph 消息状态。 |
| `integrated_agent` | Agent 实例 | 综合实战 Agent | 同时接入新闻搜索、天气 MCP、文件 MCP、记忆、安全提示和 HITL 审批，完成查新闻、查天气、整理文件的完整工作流。 |

## 13. 学习顺序建议

&emsp;&emsp;如果是第一次学习 LangChain Agent，建议按下面顺序理解这些 API：

1. 先掌握 `init_chat_model()`、`invoke()`、`stream()`、`HumanMessage` / `AIMessage` / `SystemMessage`，理解“模型调用”的基本形态。
2. 再学习 `create_agent()`、`TavilySearch()`、`@tool`，理解 Agent 如何选择和调用工具。
3. 然后学习 `MultiServerMCPClient()`、`get_tools()` 和本课程的 MCP 工具，理解外部工具服务如何接入 Agent。
4. 接着学习 `InMemorySaver()`、`thread_id` 和 `agent.get_state()`，理解多轮对话记忆。
5. 最后学习 `wrap_model_call`、`before_model`、`after_model`、`SummarizationMiddleware`、`HumanInTheLoopMiddleware` 和 `Command(resume=...)`，理解如何在 Agent Runtime 中加入工程控制逻辑。
